# 3D Gas Dispersion Model (Google Colab)

**A research and screening tool for estimating the atmospheric dispersion of single-component gases and fixed-composition gas mixtures.**

## Model description

The model links a user-defined gas release to atmospheric transport, dispersion, concentration prediction, receptor analysis, and preliminary flammability/toxic-exposure screening. It is implemented as an interactive Python notebook for Google Colab. Scenario inputs are collected in one configuration cell, while the calculation routines, chemical-property library, numerical diagnostics, plots, and data exports are arranged in separate sections.

The model provides two alternative atmospheric-dispersion pathways:

1. **Passive Gaussian dispersion.** A reflected-ground Gaussian plume represents a steady continuous source. A finite-duration Gaussian cloud formulation represents transient releases by integrating the longitudinal concentration kernel over the source-on interval. The calculation can use a constant direct mass rate, an initial gas-orifice discharge rate, or a time-varying ideal-gas vessel blowdown history.
2. **Dense-gas dispersion.** An optional DEGADISx interface calls the Python implementation of the EPA DEGADIS 2.1 model for suitable ground-level, low-momentum area sources over flat terrain. DEGADISx calculates a downwind dense-gas profile. The notebook maps the reported centreline concentration, core width, and lateral/vertical dispersion scales to a Cartesian grid for visualisation and receptor interpolation. This reconstructed display field is not a native 3D DEGADIS solution or CFD calculation.

### Main model capabilities

- **Chemical selection and mixtures:** select one or more of 28 curated chemical records and specify mole or mass fractions. The model converts between fraction bases and calculates mixture molecular weight, ideal-gas heat capacity, heat-capacity ratio, and specific gas constant.
- **Gas-release source terms:** specify a direct mass rate; estimate an initial single-phase ideal-gas orifice discharge; or calculate time-dependent pressure, temperature, inventory, and discharge from a rigid, adiabatic, well-mixed gas vessel.
- **Atmospheric meteorology:** define reference wind speed and height, wind direction, ambient temperature and pressure, Pasquill stability class, terrain roughness, and either a stability-dependent power-law wind profile or an optional Monin–Obukhov-type profile.
- **Spatial discretisation:** use a stretched, nonuniform three-dimensional grid with refinement near the source, plume centreline, and ground. The source plane, source crosswind coordinate, release elevation, and receptor heights are represented in the coordinate arrays.
- **Transient analysis:** calculate a concentration snapshot at a selected time and concentration histories at specified receptors. During an active finite-duration release, the integrated source interval can produce concentration at the source plane; after shutoff, the cloud is expected to advect away from the source.
- **Hazard screening:** display component-specific LFL/UFL reference contours, an indicative mixed-fuel LFL/UFL using Le Chatelier's rule where all required component data are available, Fuller-Schettler-Giddings estimates of binary gas-in-air molecular diffusivity, and available AEGL-2 (60 min), ERPG-2, and IDLH endpoints.
- **Visualisation and export:** provide 3D concentration isosurfaces, ground/receptor-height contours, vertical plume-centreline cross-sections, receptor time histories, numerical diagnostics, and optional CSV exports of the 3D field, source history, and selected chemical/hazard data.

### Model calculation sequence

The calculation proceeds from chemical-mixture properties and the source-term history to meteorology and dispersion. The selected dispersion engine then calculates concentration as a function of position and, for transient cases, time. Total mixture concentration is partitioned into component concentrations using the assumed fixed mixture composition. Hazard-screening routines compare those component concentrations with database reference values; they do not simulate combustion, toxicological response, or injury probability.

### Intended applications

The model is intended for research, education, scenario comparison, preliminary consequence screening, and model-development work. It can be used to investigate the influence of source rate, release duration, vessel inventory, wind speed, atmospheric stability, surface roughness, chemical composition, and receptor position on predicted gas concentrations.

### Scope and safety boundary

This is not a validated replacement for PHAST, ALOHA, DEGADIS, or CFD. The Gaussian pathway is intended for passive gas dispersion and does not represent dense-gas slumping, high-momentum jets, or detailed near-field flow. The DEGADISx pathway is restricted to a resolved ground-level, low-momentum area source over flat terrain; upstream jet release, flashing, liquid-droplet transport, aerosol rainout, and pool evaporation must be resolved separately. Its 3D field is a post-processing reconstruction of DEGADISx downwind profiles.

The vessel model assumes a single-phase ideal gas or fixed-composition ideal-gas mixture in a rigid, adiabatic, well-mixed vessel. It does not model real-gas effects, vessel-wall heat transfer, liquefied-gas equilibrium, flashing, two-phase critical flow, aerosols, composition fractionation, pipe friction, chemical reaction, terrain/building steering, or time-varying wind direction. Mixture heat capacities are treated as constant values near 298.15 K. Toxicity contour overlays are instantaneous comparisons unless a receptor exposure-window calculation is specifically performed with a sufficiently long time history. Blank chemical-database fields mean that a numerical property was not populated in the curated source set; they do not mean zero or safe.

Do not use this notebook by itself to establish emergency exclusion distances, emergency-response distances, or safety-system design limits. Independent validation and scenario-specific engineering review are required.

### Running the model

**Normal scenario edits are made in Section 2 — INPUT CONFIGURATION.** Select the chemical mixture, source model, meteorology, computational domain, receptors, and plotting options, then run the notebook from top to bottom. The default Gaussian pathway runs without the optional dense-gas package. To use the dense-gas pathway, set `CONFIG["model_engine"] = "degadisx_ground"`; the Colab runtime requires internet access to install the pinned `degadisx==0.1.4` package.


## 1. Install / import libraries

Google Colab normally includes these packages. Uncomment the installation line only if a package is missing.

The optional heavy-gas backend is installed automatically only when `CONFIG["model_engine"] = "degadisx_ground"`:

```python
%pip install degadisx==0.1.4
```

DEGADISx is a beta, research-oriented Python reimplementation of EPA DEGADIS 2.1. The Colab runtime needs internet access to install it.


In [ ]:
# %pip -q install numpy pandas scipy plotly

import math
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.special import erf
from IPython.display import display

R_UNIVERSAL = 8.314462618  # J/(mol K)
G = 9.80665                # m/s2
KAPPA = 0.40               # von Karman constant

print("Libraries loaded.")

# Curated gas-phase chemical property database.
# Molecular weights and ideal-gas Cp values are referenced to NIST Chemistry WebBook
# (SRD 69), with Cp values at/near 298.15 K. These are engineering screening values,
# not a substitute for temperature-dependent property packages in high-accuracy work.
CHEMICAL_DATABASE = {
    "Hydrogen (H2)": {"formula": "H2", "cas": "1333-74-0", "mw_g_mol": 2.01588, "cp_j_mol_k": 28.8373, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C1333740&Mask=1&Table=on&Type=JANAFG"},
    "Nitrogen (N2)": {"formula": "N2", "cas": "7727-37-9", "mw_g_mol": 28.0134, "cp_j_mol_k": 29.1238, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7727379&Mask=1&Table=on&Type=JANAFG"},
    "Oxygen (O2)": {"formula": "O2", "cas": "7782-44-7", "mw_g_mol": 31.9988, "cp_j_mol_k": 29.3826, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7782447&Mask=1&Table=on&Type=JANAFG"},
    "Argon (Ar)": {"formula": "Ar", "cas": "7440-37-1", "mw_g_mol": 39.948, "cp_j_mol_k": 20.7860, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7440371"},
    "Carbon monoxide (CO)": {"formula": "CO", "cas": "630-08-0", "mw_g_mol": 28.0101, "cp_j_mol_k": 29.14, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C630080&Mask=1&Table=on&Type=JANAFG"},
    "Carbon dioxide (CO2)": {"formula": "CO2", "cas": "124-38-9", "mw_g_mol": 44.0095, "cp_j_mol_k": 37.1300, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C124389&Mask=1&Table=on&Type=JANAFG"},
    "Methane (CH4)": {"formula": "CH4", "cas": "74-82-8", "mw_g_mol": 16.0425, "cp_j_mol_k": 35.6900, "source": "https://webbook.nist.gov/cgi/cbook.cgi?Name=methane&cTG=on"},
    "Ethane (C2H6)": {"formula": "C2H6", "cas": "74-84-0", "mw_g_mol": 30.0690, "cp_j_mol_k": 52.5100, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C74840&Mask=D&Units=CAL"},
    "Ethylene (C2H4)": {"formula": "C2H4", "cas": "74-85-1", "mw_g_mol": 28.0532, "cp_j_mol_k": 42.8540, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C74851&Table=on&Type=JANAFG"},
    "Propane (C3H8)": {"formula": "C3H8", "cas": "74-98-6", "mw_g_mol": 44.0956, "cp_j_mol_k": 73.6000, "source": "https://webbook.nist.gov/cgi/cbook.cgi?Name=propane&cTC=on&cTG=on&cTP=on"},
    "n-Butane (n-C4H10)": {"formula": "C4H10", "cas": "106-97-8", "mw_g_mol": 58.1222, "cp_j_mol_k": 98.4900, "source": "https://webbook.nist.gov/cgi/cbook.cgi?Name=butane&cTG=on"},
    "Dimethyl ether (DME)": {"formula": "C2H6O", "cas": "115-10-6", "mw_g_mol": 46.0684, "cp_j_mol_k": 65.5700, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C115106&Mask=1"},
    "Ammonia (NH3)": {"formula": "NH3", "cas": "7664-41-7", "mw_g_mol": 17.0305, "cp_j_mol_k": 35.6471, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7664417&Mask=1&Table=on&Type=JANAFG"},
    "Hydrogen sulfide (H2S)": {"formula": "H2S", "cas": "7783-06-4", "mw_g_mol": 34.0810, "cp_j_mol_k": 34.1973, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7783064&Mask=1&Table=on&Type=JANAFG"},
    "Sulfur dioxide (SO2)": {"formula": "SO2", "cas": "7446-09-5", "mw_g_mol": 64.0660, "cp_j_mol_k": 39.8736, "source": "https://webbook.nist.gov/cgi/cbook.cgi?Formula=SO2&NoIon=on&cTG=on&cTP=on"},
    "Water vapor (H2O)": {"formula": "H2O", "cas": "7732-18-5", "mw_g_mol": 18.0153, "cp_j_mol_k": 33.58, "source": "https://webbook.nist.gov/cgi/cbook.cgi?Name=water&cTG=on"},
    "Chlorine (Cl2)": {"formula": "Cl2", "cas": "7782-50-5", "mw_g_mol": 70.9060, "cp_j_mol_k": 33.95, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7782505&Mask=1&Table=on&Type=JANAFG"},
    "Isobutane (i-C4H10)": {"formula": "C4H10", "cas": "75-28-5", "mw_g_mol": 58.1222, "cp_j_mol_k": 96.65, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C75285&Mask=1&Table=on&Type=JANAFG"},
    "Ethylene oxide (C2H4O)": {"formula": "C2H4O", "cas": "75-21-8", "mw_g_mol": 44.0526, "cp_j_mol_k": 47.85, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C75218&Mask=1&Table=on&Type=JANAFG"},
    "Hydrogen cyanide (HCN)": {"formula": "HCN", "cas": "74-90-8", "mw_g_mol": 27.0253, "cp_j_mol_k": 35.85, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C74908&Mask=1&Table=on&Type=JANAFG"},
    "Nitrogen dioxide (NO2)": {"formula": "NO2", "cas": "10102-44-0", "mw_g_mol": 46.0055, "cp_j_mol_k": 36.97, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C10102440&Mask=1&Table=on&Type=JANAFG"},
    "Hydrogen chloride (HCl)": {"formula": "HCl", "cas": "7647-01-0", "mw_g_mol": 36.4609, "cp_j_mol_k": 29.136, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7647010&Mask=1&Table=on&Type=JANAFG"},
    "Carbon disulfide (CS2)": {"formula": "CS2", "cas": "75-15-0", "mw_g_mol": 76.1410, "cp_j_mol_k": 45.66, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C75150&Mask=1&Table=on&Type=JANAFG"},
    "1,3-Butadiene (C4H6)": {"formula": "C4H6", "cas": "106-99-0", "mw_g_mol": 54.0904, "cp_j_mol_k": 79.81, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C106990&Mask=1"},
    "Methyl mercaptan (CH3SH)": {"formula": "CH4S", "cas": "74-93-1", "mw_g_mol": 48.107, "cp_j_mol_k": 50.26, "source": "https://cccbdb.nist.gov/exp2x.asp?casno=74931&charge=0"},
    "Phosphine (PH3)": {"formula": "PH3", "cas": "7803-51-2", "mw_g_mol": 33.99758, "cp_j_mol_k": 37.11, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7803512&Table=on&Type=JANAFG"},
    "Phosgene (COCl2)": {"formula": "CCl2O", "cas": "75-44-5", "mw_g_mol": 98.916, "cp_j_mol_k": 57.68, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C75445&Table=on&Type=JANAFG"},
    "Nitric oxide (NO)": {"formula": "NO", "cas": "10102-43-9", "mw_g_mol": 30.0061, "cp_j_mol_k": 29.86, "source": "https://webbook.nist.gov/cgi/cbook.cgi?ID=C10102439&Table=on&Type=JANAFG"},
}


def calculate_gas_mixture(mixture_cfg):
    """Calculate ideal-gas mixture MW, Cp, gamma, Rspec and species fractions.

    `fraction_basis` is 'mole' or 'mass'. Input fractions may sum to 1.0 or 100.
    Ideal-gas Cp values are treated as constant at 298.15 K for this screening model.
    """
    basis = str(mixture_cfg.get("fraction_basis", "mole")).strip().lower()
    if basis not in {"mole", "mass"}:
        raise ValueError("mixture.fraction_basis must be 'mole' or 'mass'.")
    components = mixture_cfg.get("components", [])
    if not components:
        raise ValueError("Select at least one chemical in mixture.components.")
    names = [c.get("chemical") for c in components]
    if len(names) != len(set(names)):
        raise ValueError("A chemical may appear only once in the mixture list.")

    fractions = np.asarray([float(c.get("fraction", 0.0)) for c in components], dtype=float)
    if np.any(~np.isfinite(fractions)) or np.any(fractions < 0):
        raise ValueError("Mixture fractions must be finite and nonnegative.")
    total = float(np.sum(fractions))
    if total <= 0 or not (np.isclose(total, 1.0, atol=1e-6) or np.isclose(total, 100.0, atol=1e-4)):
        raise ValueError(f"Mixture fractions must sum to 1.0 or 100; received {total:.8g}.")
    fractions = fractions / total

    records = []
    for comp, frac in zip(components, fractions):
        name = comp.get("chemical")
        if name not in CHEMICAL_DATABASE:
            options = ", ".join(CHEMICAL_DATABASE.keys())
            raise KeyError(f"'{name}' is not in CHEMICAL_DATABASE. Available chemicals: {options}")
        rec = CHEMICAL_DATABASE[name]
        if frac > 0:
            records.append((name, rec, float(frac)))
    if not records:
        raise ValueError("At least one mixture component must have a positive fraction.")

    mw = np.asarray([r[1]["mw_g_mol"] for r in records], dtype=float)
    cp = np.asarray([r[1]["cp_j_mol_k"] for r in records], dtype=float)
    f = np.asarray([r[2] for r in records], dtype=float)
    if basis == "mole":
        y = f / f.sum()
        mw_mix = float(np.sum(y * mw))
        w = y * mw / mw_mix
    else:
        w = f / f.sum()
        inv_mw = float(np.sum(w / mw))
        mw_mix = 1.0 / inv_mw
        y = (w / mw) / inv_mw

    cp_mix = float(np.sum(y * cp))  # J/(mol K), ideal-gas mole-weighted Cp
    cv_mix = cp_mix - R_UNIVERSAL
    if cv_mix <= 0:
        raise ValueError("Calculated mixture Cv is not positive; check component data.")
    gamma_mix = cp_mix / cv_mix
    rows = []
    for (name, rec, input_fraction), yi, wi in zip(records, y, w):
        rows.append({
            "Chemical": name, "Formula": rec["formula"], "CAS": rec["cas"],
            f"Input {basis} fraction (%)": 100.0 * input_fraction,
            "Mole fraction (%)": 100.0 * yi,
            "Mass fraction (%)": 100.0 * wi,
            "MW (g/mol)": rec["mw_g_mol"],
            "Cp,gas at 298.15 K (J/mol/K)": rec["cp_j_mol_k"],
            "Property source": rec["source"],
        })
    composition = pd.DataFrame(rows)
    display_name = " + ".join(f"{name} ({100*yi:.3g} mol%)" for (name, _, _), yi in zip(records, y))
    return {
        "name": display_name,
        "molecular_weight_g_mol": mw_mix,
        "cp_molar_j_mol_k": cp_mix,
        "cv_molar_j_mol_k": cv_mix,
        "heat_capacity_ratio_gamma": gamma_mix,
        "specific_gas_constant_j_kg_k": R_UNIVERSAL / (mw_mix / 1000.0),
        "composition_mole_fractions": {name: float(yi) for (name, _, _), yi in zip(records, y)},
        "composition_mass_fractions": {name: float(wi) for (name, _, _), wi in zip(records, w)},
        "composition_table": composition,
        "property_reference_temperature_K": 298.15,
        "property_model": "Ideal-gas Cp at 298.15 K; constant-Cp mixture approximation",
    }


def get_source_gamma(cfg, source_block):
    """Use a user override if provided; otherwise use gamma calculated from mixture Cp."""
    block = cfg["source"][source_block]
    override = block.get("heat_capacity_ratio_gamma_override", None)
    if override is not None:
        val = float(override)
        if val <= 1.0:
            raise ValueError("Heat-capacity-ratio override gamma must be > 1.")
        return val
    return float(cfg["gas"]["heat_capacity_ratio_gamma"])


# -----------------------------------------------------------------------------
# Component flammability, toxicity, and transport-property records.
# Missing values are represented by None (never silently treated as zero).
# Flammability limits are volume % in air near ambient conditions unless stated.
# Toxic endpoints retain the source's exposure period and interpretation.
# -----------------------------------------------------------------------------
HAZARD_DATABASE = {
    "Hydrogen (H2)": {
        "lfl_vol_pct": 4.0, "ufl_vol_pct": 75.0,
        "flammability_status": "flammable gas",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/8729",
        "diffusion_volume_cm3_mol": 6.12,
        "diffusion_volume_basis": "tabulated Fuller volume for H2",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None,
        "idlh_basis": None, "toxicity_source": "https://cameochemicals.noaa.gov/chemical/8729",
        "hazard_note": "No AEGL/ERPG/IDLH value was listed in the cited CAMEO entry. Hydrogen is an asphyxiant and a highly flammable gas."
    },
    "Nitrogen (N2)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable; simple asphyxiant",
        "flammability_source": "https://www.cdc.gov/niosh/npg/",
        "diffusion_volume_cm3_mol": 18.5, "diffusion_volume_basis": "tabulated Fuller volume for N2",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None, "idlh_basis": None,
        "toxicity_source": None, "hazard_note": "Simple asphyxiant; no chemical-specific acute toxic endpoint entered."
    },
    "Oxygen (O2)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "oxidizer; not a fuel gas",
        "flammability_source": "https://www.cdc.gov/niosh/npg/",
        "diffusion_volume_cm3_mol": 16.3, "diffusion_volume_basis": "tabulated Fuller volume for O2",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None, "idlh_basis": None,
        "toxicity_source": None, "hazard_note": "Oxygen is an oxidizer and can intensify combustion; it is not treated as a fuel component in Le Chatelier calculations."
    },
    "Argon (Ar)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable; simple asphyxiant",
        "flammability_source": None,
        "diffusion_volume_cm3_mol": 16.2, "diffusion_volume_basis": "tabulated Fuller volume for Ar",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None, "idlh_basis": None,
        "toxicity_source": None, "hazard_note": "Simple asphyxiant; no chemical-specific acute toxic endpoint entered."
    },
    "Carbon monoxide (CO)": {
        "lfl_vol_pct": 12.5, "ufl_vol_pct": 74.0,
        "flammability_status": "flammable gas",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0105.html",
        "diffusion_volume_cm3_mol": 18.0, "diffusion_volume_basis": "tabulated Fuller volume for CO",
        "aegl2_60min_ppm": 83.0, "erpg2_ppm": 350.0, "idlh_ppm": 1200.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://www.epa.gov/aegl/carbon-monoxide-results-aegl-program",
        "hazard_note": "AEGL-2 is for 60 min. ERPG-2 is a 1-hour guideline. IDLH from NIOSH."
    },
    "Carbon dioxide (CO2)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0103.html",
        "diffusion_volume_cm3_mol": 26.9, "diffusion_volume_basis": "tabulated Fuller volume for CO2",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": 40000.0,
        "idlh_basis": "acute inhalation toxicity / asphyxiation",
        "toxicity_source": "https://www.cdc.gov/niosh/npg/npgd0103.html",
        "hazard_note": "IDLH = 40,000 ppm. CO2 can cause hypercapnia and asphyxiation; no AEGL-2/ERPG-2 value entered here."
    },
    "Methane (CH4)": {
        "lfl_vol_pct": 5.0, "ufl_vol_pct": 15.0,
        "flammability_status": "flammable gas",
        "flammability_source": "https://www.cdc.gov/niosh/docs/2016-108/pdfs/2016-108.pdf",
        "diffusion_volume_cm3_mol": 25.14, "diffusion_volume_basis": "estimated Fuller group contribution (C + H)",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": 5000.0,
        "idlh_basis": "10% LEL criterion; not a toxicological endpoint",
        "toxicity_source": "https://www.cdc.gov/niosh/docs/2016-108/pdfs/2016-108.pdf",
        "hazard_note": "NIOSH-listed IDLH is 10% LEL. Do not interpret this IDLH as an acute toxicity threshold. Methane is a simple asphyxiant."
    },
    "Ethane (C2H6)": {
        "lfl_vol_pct": 3.0, "ufl_vol_pct": 12.5,
        "flammability_status": "flammable gas",
        "flammability_source": "https://www.cdc.gov/niosh/docs/2016-108/pdfs/2016-108.pdf",
        "diffusion_volume_cm3_mol": 45.66, "diffusion_volume_basis": "estimated Fuller group contribution (C + H)",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": 3000.0,
        "idlh_basis": "10% LEL criterion; not a toxicological endpoint",
        "toxicity_source": "https://www.cdc.gov/niosh/docs/2016-108/pdfs/2016-108.pdf",
        "hazard_note": "NIOSH-listed IDLH is 10% LEL. Do not interpret this IDLH as an acute toxicity threshold. Ethane is a simple asphyxiant."
    },
    "Ethylene (C2H4)": {
        "lfl_vol_pct": 2.75, "ufl_vol_pct": 28.6,
        "flammability_status": "flammable gas",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/8655",
        "diffusion_volume_cm3_mol": 41.04, "diffusion_volume_basis": "estimated Fuller group contribution (C + H)",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None,
        "idlh_basis": None, "toxicity_source": "https://cameochemicals.noaa.gov/chemical/8655",
        "hazard_note": "No AEGL/ERPG/IDLH value entered from the cited CAMEO entry."
    },
    "Propane (C3H8)": {
        "lfl_vol_pct": 2.1, "ufl_vol_pct": 9.5,
        "flammability_status": "flammable gas",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0524.html",
        "diffusion_volume_cm3_mol": 66.18, "diffusion_volume_basis": "estimated Fuller group contribution (C + H)",
        "aegl2_60min_ppm": 17000.0, "erpg2_ppm": None, "idlh_ppm": 2100.0,
        "idlh_basis": "10% LEL criterion; not a toxicological endpoint",
        "toxicity_source": "https://www.epa.gov/aegl/propane-results-aegl-program",
        "hazard_note": "EPA flags AEGL-2 (17,000 ppm) above 50% of its stated LEL; explosion hazard must be considered. NIOSH IDLH is based on 10% LEL, not toxicity."
    },
    "n-Butane (n-C4H10)": {
        "lfl_vol_pct": 1.6, "ufl_vol_pct": 8.4,
        "flammability_status": "flammable gas",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0068.html",
        "diffusion_volume_cm3_mol": 86.70, "diffusion_volume_basis": "estimated Fuller group contribution (C + H)",
        "aegl2_60min_ppm": 17000.0, "erpg2_ppm": None, "idlh_ppm": 1600.0,
        "idlh_basis": "10% LEL criterion; not a toxicological endpoint",
        "toxicity_source": "https://www.epa.gov/aegl/butane-results-aegl-program",
        "hazard_note": "EPA flags AEGL-2 (17,000 ppm for 30 min to 8 h) above 50% of its stated LEL; explosion hazard must be considered. NIOSH IDLH is based on 10% LEL, not toxicity."
    },
    "Dimethyl ether (DME)": {
        "lfl_vol_pct": 2.0, "ufl_vol_pct": 50.0,
        "flammability_status": "flammable gas",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/585",
        "diffusion_volume_cm3_mol": 51.77, "diffusion_volume_basis": "estimated Fuller group contribution (C + H + O)",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None,
        "idlh_basis": None, "toxicity_source": "https://cameochemicals.noaa.gov/chemical/585",
        "hazard_note": "CAMEO/USCG source reports LEL 2 vol% and UEL 50 vol%. Other compilations (for example NFPA-derived HSDB data surfaced through PubChem) report 3.4/27 vol%; this source discrepancy materially affects the contour. No AEGL/ERPG/IDLH value was available in the cited CAMEO entry. Select and document the source appropriate to the study."
    },
    "Ammonia (NH3)": {
        "lfl_vol_pct": 15.0, "ufl_vol_pct": 28.0,
        "flammability_status": "flammable under some conditions; oxidizer interactions matter",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0028.html",
        "diffusion_volume_cm3_mol": 20.7, "diffusion_volume_basis": "tabulated Fuller volume for NH3",
        "aegl2_60min_ppm": 160.0, "erpg2_ppm": 150.0, "idlh_ppm": 300.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/4860",
        "hazard_note": "AEGL-2 is for 60 min. ERPG-2 is a 1-hour guideline. NIOSH notes NH3 should be treated as flammable despite its DOT transport classification."
    },
    "Hydrogen sulfide (H2S)": {
        "lfl_vol_pct": 4.0, "ufl_vol_pct": 44.0,
        "flammability_status": "flammable gas; highly toxic",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0337.html",
        "diffusion_volume_cm3_mol": 27.52, "diffusion_volume_basis": "estimated Fuller group contribution (H + S)",
        "aegl2_60min_ppm": 27.0, "erpg2_ppm": 30.0, "idlh_ppm": 100.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://www.epa.gov/aegl/hydrogen-sulfide-results-aegl-program",
        "hazard_note": "AEGL-2 is for 60 min. ERPG-2 is a 1-hour guideline. NIOSH cautions that odor is not a reliable warning because olfactory fatigue occurs."
    },
    "Sulfur dioxide (SO2)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0575.html",
        "diffusion_volume_cm3_mol": 41.8, "diffusion_volume_basis": "tabulated Fuller volume for SO2",
        "aegl2_60min_ppm": 0.75, "erpg2_ppm": 3.0, "idlh_ppm": 100.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/1554",
        "hazard_note": "AEGL-2 is for 60 min. ERPG-2 is a 1-hour guideline."
    },
    "Water vapor (H2O)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable",
        "flammability_source": None,
        "diffusion_volume_cm3_mol": 13.1, "diffusion_volume_basis": "tabulated Fuller volume for H2O",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None,
        "idlh_basis": None, "toxicity_source": None, "hazard_note": "Water vapour is not treated as a fuel or as a toxic endpoint in this starter database."
    },
    "Chlorine (Cl2)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable oxidizer",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0115.html",
        "diffusion_volume_cm3_mol": 38.4, "diffusion_volume_basis": "tabulated Fuller volume for Cl2",
        "aegl2_60min_ppm": 2.0, "erpg2_ppm": 3.0, "idlh_ppm": 10.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/2862",
        "hazard_note": "AEGL-2 is for 60 min. ERPG-2 is a 1-hour guideline. Chlorine is a strong oxidizer but is not itself a fuel gas."
    },
    "Isobutane (i-C4H10)": {
        "lfl_vol_pct": 1.8, "ufl_vol_pct": 8.4, "flammability_status": "flammable gas",
        "flammability_source": "https://m.cameochemicals.noaa.gov/chemical/8744",
        "diffusion_volume_cm3_mol": 85.8, "diffusion_volume_basis": "Fuller atomic-increment estimate; not a measured binary diffusion volume",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": None, "idlh_basis": None,
        "toxicity_source": "https://m.cameochemicals.noaa.gov/chemical/8744",
        "hazard_note": "CAMEO reports a flammable gas; no chemical-specific AEGL-2, ERPG-2 or IDLH acute-toxicity endpoint entered in this database."
    },
    "Ethylene oxide (C2H4O)": {
        "lfl_vol_pct": 3.0, "ufl_vol_pct": 100.0, "flammability_status": "extremely flammable gas",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/694",
        "diffusion_volume_cm3_mol": 46.4, "diffusion_volume_basis": "Fuller atomic-increment estimate; not a measured binary diffusion volume",
        "aegl2_60min_ppm": 45.0, "erpg2_ppm": 50.0, "idlh_ppm": 800.0,
        "idlh_basis": "NIOSH IDLH; potential occupational carcinogen noted by NIOSH",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/694",
        "hazard_note": "High consequence flammable and toxic gas. Verify current endpoint versions and chemical-specific handling before use."
    },
    "Hydrogen cyanide (HCN)": {
        "lfl_vol_pct": 5.6, "ufl_vol_pct": 40.0, "flammability_status": "flammable gas; acutely toxic",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/14746",
        "diffusion_volume_cm3_mol": 24.17, "diffusion_volume_basis": "Fuller atomic-increment estimate; not a measured binary diffusion volume",
        "aegl2_60min_ppm": 7.1, "erpg2_ppm": 10.0, "idlh_ppm": 50.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/14746",
        "hazard_note": "Highly toxic by inhalation. AEGL-2 is 60 min; ERPG-2 is 1 h; IDLH is a 30-min escape-planning reference."
    },
    "Nitrogen dioxide (NO2)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable oxidizing gas",
        "flammability_source": "https://cameochemicals.noaa.gov/report?key=CH4072",
        "diffusion_volume_cm3_mol": 16.65, "diffusion_volume_basis": "Fuller atomic-increment estimate; not a measured binary diffusion volume",
        "aegl2_60min_ppm": 12.0, "erpg2_ppm": 15.0, "idlh_ppm": 13.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/report?key=CH4072",
        "hazard_note": "Toxic oxidizing gas; not a fuel. AEGL-2 is 60 min; ERPG-2 is 1 h."
    },
    "Hydrogen chloride (HCl)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable corrosive gas",
        "flammability_source": "https://m.cameochemicals.noaa.gov/chemical/8731",
        "diffusion_volume_cm3_mol": 21.48, "diffusion_volume_basis": "Fuller atomic-increment estimate; not a measured binary diffusion volume",
        "aegl2_60min_ppm": 22.0, "erpg2_ppm": 20.0, "idlh_ppm": 50.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://m.cameochemicals.noaa.gov/chemical/8731",
        "hazard_note": "Anhydrous hydrogen chloride gas. Do not confuse gas data with hydrochloric-acid solution data."
    },
    "Carbon disulfide (CS2)": {
        "lfl_vol_pct": 1.0, "ufl_vol_pct": 50.0, "flammability_status": "extremely flammable volatile liquid/vapour",
        "flammability_source": "https://m.cameochemicals.noaa.gov/chemical/2813",
        "diffusion_volume_cm3_mol": 50.5, "diffusion_volume_basis": "Fuller atomic-increment estimate; not a measured binary diffusion volume",
        "aegl2_60min_ppm": 160.0, "erpg2_ppm": 50.0, "idlh_ppm": 500.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://m.cameochemicals.noaa.gov/chemical/2813",
        "hazard_note": "Highly flammable and toxic vapour. The volatile-liquid source term is not represented by the current gas-only release module."
    },
    "1,3-Butadiene (C4H6)": {
        "lfl_vol_pct": 2.0, "ufl_vol_pct": 11.5, "flammability_status": "highly flammable gas; polymerizable; potential occupational carcinogen",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/4891",
        "diffusion_volume_cm3_mol": 77.88, "diffusion_volume_basis": "Fuller atomic-increment estimate (C/H); not a measured binary diffusion volume",
        "aegl2_60min_ppm": 5300.0, "erpg2_ppm": 500.0, "idlh_ppm": 2000.0,
        "idlh_basis": "10% LEL criterion; not a toxicological endpoint",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/4891",
        "hazard_note": "Interim AEGL-2 (60 min) is flagged by NOAA CAMEO as 10–49% of LEL; explosion hazard must also be considered. IDLH 2,000 ppm is based on 10% LEL and is excluded from toxic endpoint screening. Shipped as liquefied gas; polymerization and liquid release are not modeled here."
    },
    "Methyl mercaptan (CH3SH)": {
        "lfl_vol_pct": 3.9, "ufl_vol_pct": 21.8, "flammability_status": "highly flammable and acutely toxic gas/vapour",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/3950",
        "diffusion_volume_cm3_mol": 41.42, "diffusion_volume_basis": "Fuller atomic-increment estimate (C/H/S); not a measured binary diffusion volume",
        "aegl2_60min_ppm": 23.0, "erpg2_ppm": 30.0, "idlh_ppm": 150.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/3950",
        "hazard_note": "NOAA CAMEO identifies it as poisonous and flammable; odor is not a reliable warning. Low-boiling liquid/vapour behaviour and liquid spill evaporation are not modeled by the gas-only source term."
    },
    "Phosphine (PH3)": {
        "lfl_vol_pct": 1.79, "ufl_vol_pct": None, "flammability_status": "pyrophoric / spontaneously igniting; numeric UFL unavailable in cited CAMEO record",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/1322",
        "diffusion_volume_cm3_mol": None, "diffusion_volume_basis": "Not entered: no validated Fuller volume identified in the curated source set",
        "aegl2_60min_ppm": 2.0, "erpg2_ppm": 0.5, "idlh_ppm": 50.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/1322",
        "hazard_note": "Highly toxic and pyrophoric. CAMEO reports LEL 1.79% but does not provide a numeric UEL; mixed-fuel Le Chatelier limits are therefore disabled when this species is present. No Fuller volume is supplied, so D in air is left missing rather than guessed."
    },
    "Phosgene (COCl2)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable; extremely toxic and corrosive",
        "flammability_source": "https://cameochemicals.noaa.gov/chemical/4228",
        "diffusion_volume_cm3_mol": 60.98, "diffusion_volume_basis": "Fuller atomic-increment estimate (C/Cl/O); not a measured binary diffusion volume",
        "aegl2_60min_ppm": 0.3, "erpg2_ppm": 0.5, "idlh_ppm": 2.0,
        "idlh_basis": "acute inhalation toxicity",
        "toxicity_source": "https://cameochemicals.noaa.gov/chemical/4228",
        "hazard_note": "Extremely toxic choking agent. Gas or low-boiling volatile liquid; this database entry is for the gas-phase component and the existing source term does not model liquid phosgene evaporation or aerosol formation."
    },
    "Nitric oxide (NO)": {
        "lfl_vol_pct": None, "ufl_vol_pct": None, "flammability_status": "nonflammable oxidizing gas; can accelerate combustion",
        "flammability_source": "https://www.cdc.gov/niosh/npg/npgd0448.html",
        "diffusion_volume_cm3_mol": 11.17, "diffusion_volume_basis": "Fuller atomic-increment estimate (N/O); not a measured binary diffusion volume",
        "aegl2_60min_ppm": None, "erpg2_ppm": None, "idlh_ppm": 100.0,
        "idlh_basis": "NIOSH IDLH; acute criteria and AEGL/ERPG not established in the cited sources",
        "toxicity_source": "https://www.cdc.gov/niosh/npg/npgd0448.html",
        "hazard_note": "NIOSH lists IDLH 100 ppm and reports that NO is nonflammable but accelerates combustion. The cited NIOSH page does not provide LFL/UFL. Nitric oxide is rapidly converted in air to NO2; this simple single-species dispersion model does not model that chemistry."
    },
}

# Extend (not replace) the original NIST-based thermophysical records.
for _chemical_name, _hazard_record in HAZARD_DATABASE.items():
    CHEMICAL_DATABASE[_chemical_name].update(_hazard_record)


# Source URLs are separated by endpoint family because AEGL, ERPG and IDLH values
# are not always derived from the same organization or publication.
_ENDPOINT_SOURCES = {
    "Carbon monoxide (CO)": {"aegl_source": "https://www.epa.gov/aegl/carbon-monoxide-results-aegl-program", "erpg_source": "https://cameochemicals.noaa.gov/chemical/335", "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0105.html"},
    "Carbon dioxide (CO2)": {"aegl_source": None, "erpg_source": None, "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0103.html"},
    "Methane (CH4)": {"aegl_source": None, "erpg_source": None, "idlh_source": "https://www.cdc.gov/niosh/docs/2016-108/pdfs/2016-108.pdf"},
    "Ethane (C2H6)": {"aegl_source": None, "erpg_source": None, "idlh_source": "https://www.cdc.gov/niosh/docs/2016-108/pdfs/2016-108.pdf"},
    "Propane (C3H8)": {"aegl_source": "https://www.epa.gov/aegl/propane-results-aegl-program", "erpg_source": None, "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0524.html"},
    "n-Butane (n-C4H10)": {"aegl_source": "https://www.epa.gov/aegl/butane-results-aegl-program", "erpg_source": None, "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0068.html"},
    "Ammonia (NH3)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/4860", "erpg_source": "https://cameochemicals.noaa.gov/chemical/4860", "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0028.html"},
    "Hydrogen sulfide (H2S)": {"aegl_source": "https://www.epa.gov/aegl/hydrogen-sulfide-results-aegl-program", "erpg_source": "https://cameochemicals.noaa.gov/chemical/3625", "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0337.html"},
    "Sulfur dioxide (SO2)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/1554", "erpg_source": "https://cameochemicals.noaa.gov/chemical/1554", "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0575.html"},
    "Chlorine (Cl2)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/2862", "erpg_source": "https://cameochemicals.noaa.gov/chemical/2862", "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0115.html"},
    "Isobutane (i-C4H10)": {"aegl_source": None, "erpg_source": None, "idlh_source": None},
    "Ethylene oxide (C2H4O)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/694", "erpg_source": "https://cameochemicals.noaa.gov/chemical/694", "idlh_source": "https://cameochemicals.noaa.gov/chemical/694"},
    "Hydrogen cyanide (HCN)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/14746", "erpg_source": "https://cameochemicals.noaa.gov/chemical/14746", "idlh_source": "https://cameochemicals.noaa.gov/chemical/14746"},
    "Nitrogen dioxide (NO2)": {"aegl_source": "https://cameochemicals.noaa.gov/report?key=CH4072", "erpg_source": "https://cameochemicals.noaa.gov/report?key=CH4072", "idlh_source": "https://www.cdc.gov/niosh/npg/"},
    "Hydrogen chloride (HCl)": {"aegl_source": "https://m.cameochemicals.noaa.gov/chemical/8731", "erpg_source": "https://m.cameochemicals.noaa.gov/chemical/8731", "idlh_source": "https://m.cameochemicals.noaa.gov/chemical/8731"},
    "Carbon disulfide (CS2)": {"aegl_source": "https://m.cameochemicals.noaa.gov/chemical/2813", "erpg_source": "https://m.cameochemicals.noaa.gov/chemical/2813", "idlh_source": "https://m.cameochemicals.noaa.gov/chemical/2813"},
    "1,3-Butadiene (C4H6)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/4891", "erpg_source": "https://cameochemicals.noaa.gov/chemical/4891", "idlh_source": "https://cameochemicals.noaa.gov/chemical/4891"},
    "Methyl mercaptan (CH3SH)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/3950", "erpg_source": "https://cameochemicals.noaa.gov/chemical/3950", "idlh_source": "https://cameochemicals.noaa.gov/chemical/3950"},
    "Phosphine (PH3)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/1322", "erpg_source": "https://cameochemicals.noaa.gov/chemical/1322", "idlh_source": "https://cameochemicals.noaa.gov/chemical/1322"},
    "Phosgene (COCl2)": {"aegl_source": "https://cameochemicals.noaa.gov/chemical/4228", "erpg_source": "https://cameochemicals.noaa.gov/chemical/4228", "idlh_source": "https://cameochemicals.noaa.gov/chemical/4228"},
    "Nitric oxide (NO)": {"aegl_source": None, "erpg_source": None, "idlh_source": "https://www.cdc.gov/niosh/npg/npgd0448.html"},
}
for _name, _sources in _ENDPOINT_SOURCES.items():
    CHEMICAL_DATABASE[_name].update(_sources)

FULLER_AIR_DIFFUSION_VOLUME_CM3_MOL = 19.7
FULLER_AIR_MW_G_MOL = 28.965
FULLER_REFERENCE = "Fuller-Schettler-Giddings correlation; COMSOL transport-property documentation"
FULLER_REFERENCE_URL = "https://doc.comsol.com/6.4/doc/com.comsol.help.lgp/lgp_ug_thermo.5.27.html"


def fuller_diffusivity_in_air_m2_s(chemical_record, temperature_K, pressure_Pa, air_mw_g_mol=FULLER_AIR_MW_G_MOL):
    """Estimate binary gas-in-air diffusivity by Fuller-Schettler-Giddings.

    Returns m2/s. Uses pressure in atm, molecular weights in g/mol, and Fuller
    diffusion volumes in cm3/mol. This is an estimate for dilute gas in air.
    """
    T = float(temperature_K)
    P_atm = float(pressure_Pa) / 101325.0
    Mi = float(chemical_record["mw_g_mol"])
    Vi = chemical_record.get("diffusion_volume_cm3_mol")
    Vair = FULLER_AIR_DIFFUSION_VOLUME_CM3_MOL
    if T <= 0 or P_atm <= 0 or Vi is None or Mi <= 0:
        return np.nan
    Mij = 2.0 / (1.0 / Mi + 1.0 / float(air_mw_g_mol))
    D_cm2_s = (0.00143 * T**1.75) / (
        P_atm * math.sqrt(Mij) * (float(Vi)**(1.0/3.0) + Vair**(1.0/3.0))**2
    )
    return float(D_cm2_s * 1.0e-4)


def selected_toxicity_endpoint(record, endpoint_key):
    """Return (value_ppm, exposure_window_s, display_label, note). None means unavailable."""
    if endpoint_key == "AEGL2_60min":
        value = record.get("aegl2_60min_ppm")
        return (value, 3600.0, "AEGL-2 (60 min)", record.get("hazard_note", "")) if value is not None else (None, 3600.0, "AEGL-2 (60 min)", "No AEGL-2 60-min value in the embedded database.")
    if endpoint_key == "ERPG2":
        value = record.get("erpg2_ppm")
        return (value, 3600.0, "ERPG-2 (1 h)", record.get("hazard_note", "")) if value is not None else (None, 3600.0, "ERPG-2 (1 h)", "No ERPG-2 value in the embedded database.")
    if endpoint_key == "IDLH":
        value = record.get("idlh_ppm")
        basis = str(record.get("idlh_basis") or "")
        # NIOSH IDLH values assigned as 10% LEL are not toxicity endpoints.
        if value is not None and "10% LEL" not in basis:
            return (value, 1800.0, "IDLH (30-min escape planning)", record.get("hazard_note", ""))
        if value is not None:
            return (None, 1800.0, "IDLH", "IDLH entry is based on 10% LEL, not acute toxicity; excluded from toxic contours.")
        return (None, 1800.0, "IDLH", "No IDLH value in the embedded database.")
    raise ValueError("hazard_assessment.toxicity_endpoint must be AEGL2_60min, ERPG2, or IDLH.")


def calculate_mixture_hazard_properties(mixture_cfg, gas, met):
    """Build component hazard table and Le Chatelier mixed-fuel LFL/UFL estimate."""
    rows = []
    fuel_terms_lfl = []
    fuel_terms_ufl = []
    total_fuel_mole_fraction = 0.0
    incomplete_fuel_limit_names = []
    for name, yi in gas["composition_mole_fractions"].items():
        rec = CHEMICAL_DATABASE[name]
        lfl = rec.get("lfl_vol_pct")
        ufl = rec.get("ufl_vol_pct")
        is_declared_fuel = (lfl is not None or ufl is not None or
                            "flammable" in str(rec.get("flammability_status", "")).lower() or
                            "pyrophoric" in str(rec.get("flammability_status", "")).lower() or
                            "spontaneously igniting" in str(rec.get("flammability_status", "")).lower())
        if is_declared_fuel:
            total_fuel_mole_fraction += float(yi)
            if lfl is None or ufl is None:
                incomplete_fuel_limit_names.append(name)
            else:
                fuel_terms_lfl.append((float(yi), float(lfl)))
                fuel_terms_ufl.append((float(yi), float(ufl)))
        Dair = fuller_diffusivity_in_air_m2_s(
            rec, met["temperature_K"], met["pressure_Pa"], met.get("air_molecular_weight_g_mol", FULLER_AIR_MW_G_MOL)
        )
        endpoint_specs = {}
        for key in ("AEGL2_60min", "ERPG2", "IDLH"):
            value, window_s, label, note = selected_toxicity_endpoint(rec, key)
            endpoint_specs[key] = {"value_ppm": value, "window_s": window_s, "label": label, "note": note}
        rows.append({
            "Chemical": name,
            "Mole fraction (%)": 100.0 * float(yi),
            "LFL (vol% air)": lfl,
            "UFL (vol% air)": ufl,
            "Flammability status": rec.get("flammability_status"),
            "D in air (m2/s)": Dair,
            "Fuller volume (cm3/mol)": rec.get("diffusion_volume_cm3_mol"),
            "AEGL-2 60 min (ppm)": rec.get("aegl2_60min_ppm"),
            "ERPG-2 (ppm)": rec.get("erpg2_ppm"),
            "IDLH (ppm)": rec.get("idlh_ppm"),
            "IDLH basis": rec.get("idlh_basis"),
            "Flammability source": rec.get("flammability_source"),
            "AEGL source": rec.get("aegl_source", rec.get("toxicity_source")),
            "ERPG source": rec.get("erpg_source", rec.get("toxicity_source")),
            "IDLH source": rec.get("idlh_source", rec.get("toxicity_source")),
            "Diffusivity method": FULLER_REFERENCE_URL,
            "Notes": rec.get("hazard_note"),
            "endpoint_specs": endpoint_specs,
        })
    # Le Chatelier rule: normalize only among combustible fuel components.
    if incomplete_fuel_limit_names:
        # Do not silently omit a flammable species whose numerical UFL/LFL is unavailable.
        lfl_mix = ufl_mix = None
        flammability_rule_note = (
            "Mixed-fuel LFL/UFL not calculated because numeric LFL and UFL are not both "
            "available for: " + ", ".join(incomplete_fuel_limit_names)
        )
    elif total_fuel_mole_fraction > 0 and fuel_terms_lfl:
        fuel_y = [yi / total_fuel_mole_fraction for yi, _ in fuel_terms_lfl]
        lfl_mix = 1.0 / sum(y / lfl for y, (_, lfl) in zip(fuel_y, fuel_terms_lfl))
        ufl_mix = 1.0 / sum(y / ufl for y, (_, ufl) in zip(fuel_y, fuel_terms_ufl))
        flammability_rule_note = "Le Chatelier estimate over components with numeric LFL and UFL values"
    else:
        lfl_mix = ufl_mix = None
        flammability_rule_note = "No numeric combustible-fuel LFL/UFL data available for selected components"
    return {
        "component_table": pd.DataFrame(rows),
        "component_names": list(gas["composition_mole_fractions"].keys()),
        "fuel_mole_fraction_in_released_mixture": float(total_fuel_mole_fraction),
        "mixed_fuel_lfl_vol_pct": lfl_mix,
        "mixed_fuel_ufl_vol_pct": ufl_mix,
        "mixed_fuel_lfl_ppm": None if lfl_mix is None else float(lfl_mix * 10000.0),
        "mixed_fuel_ufl_ppm": None if ufl_mix is None else float(ufl_mix * 10000.0),
        "flammability_rule": flammability_rule_note + "; excludes inerting/oxygen enrichment and interaction effects",
        "diffusivity_method": FULLER_REFERENCE,
        "diffusivity_source": FULLER_REFERENCE_URL,
    }


## 2. INPUT CONFIGURATION — edit this cell for each scenario

**Tip:** Use `model_mode="puff"` for a finite release or a vessel blowdown. Use `model_mode="continuous"` for a steady direct source. All pressures are absolute. Coordinates are local to the wind direction: `x` downwind, `y` crosswind, and `z` above ground.

For gas mixtures, edit `CONFIG["mixture"]`. Example for a 70/30 mole-percent propane/n-butane vapour mixture:

```python
"mixture": {
    "fraction_basis": "mole",
    "components": [
        {"chemical": "Propane (C3H8)", "fraction": 0.70},
        {"chemical": "n-Butane (n-C4H10)", "fraction": 0.30},
    ],
},
```

For mass fractions, change `fraction_basis` to `"mass"`; the model converts them to mole fractions internally. Fractions may be entered as decimals summing to 1.0 or percentages summing to 100.


Hazard properties are looked up automatically for each selected component. The database includes LFL/UFL where sourced, an estimated binary molecular diffusivity in air (Fuller-Schettler-Giddings), and available AEGL-2 (60 min), ERPG-2, and NIOSH IDLH values. Missing endpoints remain blank. Use `CONFIG["hazard_assessment"]["toxicity_endpoint"]` to select the toxic endpoint family for contour screening.


### Selecting the heavy-gas engine

For dense-gas screening, set `CONFIG["model_engine"] = "degadisx_ground"`. This route currently requires `source_term_mode="direct_rate"`, `release_height_m=0.0`, and a user-specified **equivalent ground-level source radius**. Do not enter an orifice diameter as the source radius. DEGADISx is not being used here to turn a raw high-pressure jet or liquefied-gas leak directly into a ground source. Flashing, jet trajectory/touchdown, aerosol rainout, and pool evaporation must be resolved separately before using this ground-layer route. The ordinary Gaussian plots remain available as a passive-model comparator.


In [ ]:
CONFIG = {
    # "gaussian" = existing passive plume/puff engine; "degadisx_ground" = dense-gas engine
    "model_engine": "gaussian",
    "model_mode": "puff",  # transient finite-duration Gaussian cloud; or "continuous" steady plume

    "source": {
        # "direct_rate", "orifice_initial", or "vessel_gas_blowdown"
        "source_term_mode": "direct_rate",
        "mass_rate_kg_s": 0.05,          # direct rate / initial orifice rate (kg/s)
        "release_duration_s": 120.0,     # duration for direct_rate / orifice_initial
        "release_height_m": 1.0,
        "source_y_m": 0.0,

        "orifice": {
            "upstream_pressure_bar_abs": 5.0,
            "hole_diameter_mm": 5.0,
            "discharge_coefficient": 0.80,
            "heat_capacity_ratio_gamma_override": None,  # None = calculate from selected mixture
            "gas_temperature_K": 298.15,
        },

        "vessel": {
            "vessel_volume_m3": 0.10,
            "initial_pressure_bar_abs": 8.0,
            "initial_temperature_K": 298.15,
            "hole_diameter_mm": 3.0,
            "discharge_coefficient": 0.80,
            "heat_capacity_ratio_gamma_override": None,  # None = calculate from selected mixture
            "maximum_release_duration_s": 1800.0,
            "profile_output_step_s": 1.0,
        },
    },

    # Select one or more species from CHEMICAL_DATABASE.
    # fraction_basis = "mole" or "mass". Fractions may sum to 1.0 or 100.
    # Example LPG-like vapour mixture: propane 0.70 + n-butane 0.30 (mole basis).
    "mixture": {
        "fraction_basis": "mole",
        "components": [
            {"chemical": "Nitrogen (N2)", "fraction": 1.0},
        ],
    },

    "hazard_assessment": {
        "plot_hazard_contours": True,
        # Select one toxicity endpoint family: "AEGL2_60min", "ERPG2", or "IDLH".
        "toxicity_endpoint": "AEGL2_60min",
        "flammability_rule": "LeChatelier",
        "snapshot_contours_are_instantaneous": True,
    },

    "meteorology": {
        "wind_speed_reference_m_s": 3.0,
        "wind_reference_height_m": 10.0,
        "wind_direction_from_deg": 0.0,   # meteorological: 0=N, 90=E
        "temperature_K": 298.15,
        "pressure_Pa": 101325.0,
        "relative_humidity_pct": 50.0,
        "air_molecular_weight_g_mol": 28.965,
        "stability_class": "D",          # A-F
        "terrain": "Rural",              # "Rural" or "Urban"
        "roughness_length_m": 0.03,       # used for roughness/stability parameters; edit for site
        "wind_profile_model": "power_law", # "power_law" (ALOHA typical) or "monin_obukhov"
        "power_law_n_override": None,      # optional site-calibrated n; otherwise stability-class default
    },

    "heavy_gas": {
        # DEGADISx is for a low-momentum ground-level area source over flat terrain.
        # Set release_height_m=0 and source_term_mode="direct_rate" to use this route.
        "source_radius_m": 1.0,          # equivalent radius of resolved ground-level gas source, NOT hole diameter
        "source_temperature_K": 298.15,
        "averaging_time_s": 600.0,       # set 10 s for flammable-cloud averaging or a justified site value
        "mole_fraction_cutoff": 1.0e-6,
        "snapshot_selection": "peak_centerline",  # "peak_centerline" or "closest_to_time"
        "snapshot_time_s": 120.0,
        "nx": 100,
        "ny": 81,
        "nz": 51,
        "y_half_width_m": 200.0,
        "z_max_m": 80.0,
        "plot_isosurface_level_ppm": 100.0,
        "run_receptor_histories": True,
    },

    "domain": {
        "x_max_m": 1200.0,
        "y_half_width_m": 200.0,
        "z_max_m": 150.0,
        "nx": 180,
        "ny": 91,
        "nz": 65,
        "x_refinement_end_m": 30.0,       # dense linear x-grid around source; far field is geometrically stretched
        "y_stretch_factor": 2.5,          # larger = more points near centreline
        "z_stretch_power": 1.6,           # larger = more points near ground
        "receptor_height_m": 1.5,
    },

    "puff": {
        "release_interval_s": 1.0,        # time bin for time-varying blowdown; constant-rate release is integrated as one exact-duration segment
        "observation_time_s": 60.0,       # default snapshot is DURING the 120 s release; change to inspect post-shutoff cloud
        "history_end_s": 600.0,
        "history_step_s": 5.0,
    },

    "plot": {
        "isosurface_level_ppm": 10.0,
        "ground_contour_level_ppm": 10.0,
        "contour_scale": "log",          # "log" or "linear"
        "log_floor_ppm": 0.01,
        "show_3d_isosurface": True,
    },

    "export": {
        "write_full_3d_csv": False,
        "csv_filename": "gas_dispersion_3d_field.csv",
        "write_release_profile_csv": False,
        "release_profile_filename": "gas_release_profile.csv",
        "write_hazard_database_csv": False,
        "hazard_database_filename": "selected_component_hazards.csv",
    },
}

# Receptors are entered in local plume coordinates (m).
RECEPTORS = [
    {"name": "Receptor 1", "x_m": 100.0, "y_m": 0.0, "z_m": 1.5},
    {"name": "Receptor 2", "x_m": 250.0, "y_m": 25.0, "z_m": 1.5},
    {"name": "Receptor 3", "x_m": 500.0, "y_m": -25.0, "z_m": 1.5},
]

# Input validation
assert CONFIG["model_engine"] in {"gaussian", "degadisx_ground"}
assert CONFIG["model_mode"] in {"continuous", "puff"}
assert CONFIG["source"]["source_term_mode"] in {"direct_rate", "orifice_initial", "vessel_gas_blowdown"}
assert CONFIG["meteorology"]["stability_class"] in list("ABCDEF")
assert CONFIG["meteorology"]["terrain"] in {"Rural", "Urban"}
assert CONFIG["meteorology"]["wind_speed_reference_m_s"] > 0
assert CONFIG["meteorology"]["wind_reference_height_m"] > CONFIG["meteorology"]["roughness_length_m"] > 0
CONFIG["gas"] = calculate_gas_mixture(CONFIG["mixture"])
CONFIG["hazard_properties"] = calculate_mixture_hazard_properties(
    CONFIG["mixture"], CONFIG["gas"], CONFIG["meteorology"]
)
assert CONFIG["gas"]["molecular_weight_g_mol"] > 0
assert CONFIG["hazard_assessment"]["toxicity_endpoint"] in {"AEGL2_60min", "ERPG2", "IDLH"}
assert CONFIG["gas"]["heat_capacity_ratio_gamma"] > 1
assert CONFIG["source"]["release_height_m"] >= 0
assert 0.0 <= CONFIG["meteorology"]["relative_humidity_pct"] <= 100.0
assert CONFIG["heavy_gas"]["source_radius_m"] > 0
assert CONFIG["heavy_gas"]["averaging_time_s"] > 0
assert CONFIG["heavy_gas"]["mole_fraction_cutoff"] > 0
assert CONFIG["heavy_gas"]["snapshot_selection"] in {"peak_centerline", "closest_to_time"}
assert CONFIG["domain"]["nx"] >= 30 and CONFIG["domain"]["ny"] >= 30 and CONFIG["domain"]["nz"] >= 20
assert CONFIG["plot"]["contour_scale"] in {"log", "linear"}
if CONFIG["source"]["source_term_mode"] == "vessel_gas_blowdown":
    assert CONFIG["model_mode"] == "puff", "Vessel blowdown is transient; set model_mode='puff'."
if CONFIG["model_mode"] == "continuous":
    assert CONFIG["source"]["source_term_mode"] == "direct_rate", "Continuous mode requires source_term_mode='direct_rate'."
print("Scenario inputs and chemical mixture are valid.")
print(f"Hazard properties calculated for {len(CONFIG['hazard_properties']['component_names'])} selected component(s).")
if CONFIG["hazard_properties"]["mixed_fuel_lfl_vol_pct"] is not None:
    print(f"Le Chatelier mixed-fuel LFL/UFL = {CONFIG['hazard_properties']['mixed_fuel_lfl_vol_pct']:.3g} / {CONFIG['hazard_properties']['mixed_fuel_ufl_vol_pct']:.3g} vol% in air.")
else:
    print("No flammable fuel components selected; mixture LFL/UFL is not defined.")
print(f"Mixture: {CONFIG['gas']['name']}")
print(f"Mixture MW: {CONFIG['gas']['molecular_weight_g_mol']:.5g} g/mol; "
      f"Cp: {CONFIG['gas']['cp_molar_j_mol_k']:.4g} J/mol/K; "
      f"gamma: {CONFIG['gas']['heat_capacity_ratio_gamma']:.5g}")
if CONFIG["model_engine"] == "degadisx_ground":
    if CONFIG["source"]["source_term_mode"] != "direct_rate":
        raise ValueError("The DEGADISx ground-area route currently requires source_term_mode='direct_rate'. Resolve jet/flash/rainout upstream and enter the resulting ground-level contaminant rate.")
    if CONFIG["source"]["release_height_m"] > 1e-6:
        raise ValueError("DEGADISx ground-area route requires source.release_height_m=0.0 m.")
    if abs(CONFIG["source"].get("source_y_m", 0.0)) > 1e-9:
        raise ValueError("DEGADISx ground-area route uses local plume coordinates; set source_y_m=0.0.")
    print("DEGADISx ground-area mode selected: confirm that the user-defined source radius/rate represent a resolved, low-momentum ground-level vapour source.")


### 2A. Chemical database and mixture composition

Choose one or more species from `CHEMICAL_DATABASE` in `CONFIG['mixture']['components']`; enter their mole or mass fractions. Fractions must sum to **1.0 or 100**. The model converts between mole and mass basis, then calculates mixture molecular weight, ideal-gas heat capacity, heat-capacity ratio (`gamma`) and specific gas constant.

The curated database contains **28 species** covering common fuel gases, DME, air components, toxic industrial gases, and other industrial chemicals. Thermophysical records include molecular weight and gas-phase heat capacity near 298.15 K, with links to NIST Chemistry WebBook/CCCBDB records. Flammability and exposure endpoints are linked to NOAA CAMEO, EPA and NIOSH sources. Missing values remain blank; no endpoint is inferred from a neighbouring chemical.

Heat capacities are gas-phase values near 298.15 K and are treated as constant in this screening mixture-property model. Fuller diffusion volumes marked as atomic-increment estimates are estimates, not measured binary diffusivities. Phosphine diffusivity is left unavailable because a defensible Fuller volume was not established in the curated source set.

**Special data handling:** the cited phosphine CAMEO record has an LFL but no numeric UFL; therefore, the model disables the mixed-fuel Le Chatelier LFL/UFL calculation whenever phosphine is present rather than silently dropping it. Nitric oxide is identified by NIOSH as nonflammable but able to accelerate combustion; it is not included as a fuel in Le Chatelier calculations. Nitric oxide is rapidly converted to NO₂ in air, a reaction not represented by this single-species model.

In [ ]:
# Browse core thermophysical data and hazard/transport properties for the selected chemicals.
chemical_database_df = pd.DataFrame([
    {"Chemical": name, "Formula": rec["formula"], "CAS": rec["cas"],
     "MW (g/mol)": rec["mw_g_mol"], "Cp,gas at 298.15 K (J/mol/K)": rec["cp_j_mol_k"],
     "LFL (vol%)": rec.get("lfl_vol_pct"), "UFL (vol%)": rec.get("ufl_vol_pct"),
     "D in air (m2/s, scenario T/P)": fuller_diffusivity_in_air_m2_s(
         rec, CONFIG["meteorology"]["temperature_K"], CONFIG["meteorology"]["pressure_Pa"],
         CONFIG["meteorology"].get("air_molecular_weight_g_mol", FULLER_AIR_MW_G_MOL)),
     "AEGL-2 60 min (ppm)": rec.get("aegl2_60min_ppm"),
     "ERPG-2 (ppm)": rec.get("erpg2_ppm"),
     "IDLH (ppm)": rec.get("idlh_ppm"), "IDLH basis": rec.get("idlh_basis"),
     "Source (flammability)": rec.get("flammability_source"),
     "Source (toxicity)": rec.get("toxicity_source"),
     "AEGL source": rec.get("aegl_source"), "ERPG source": rec.get("erpg_source"),
     "IDLH source": rec.get("idlh_source")}
    for name, rec in CHEMICAL_DATABASE.items()
])
print("Built-in chemical database (blank = no value entered in this curated version):")
display(chemical_database_df)
print("Selected mixture composition:")
display(CONFIG["gas"]["composition_table"])
print("Selected-component hazard and transport properties:")
hazard_table_for_display = CONFIG["hazard_properties"]["component_table"].drop(columns=["endpoint_specs"])
display(hazard_table_for_display)
print(f"Mixture MW = {CONFIG['gas']['molecular_weight_g_mol']:.5g} g/mol")
print(f"Mixture Cp = {CONFIG['gas']['cp_molar_j_mol_k']:.5g} J/(mol K) at 298.15 K")
print(f"Mixture gamma = {CONFIG['gas']['heat_capacity_ratio_gamma']:.5g}")
print(f"Mixture specific gas constant = {CONFIG['gas']['specific_gas_constant_j_kg_k']:.5g} J/(kg K)")
print(f"Fuller method: {FULLER_REFERENCE}; D_i-air is estimated at the scenario T/P.")

## 3. Model equations and numerical methods

### Atmospheric wind profile

The default vertical profile is the stability-dependent power law documented in ALOHA 5.4.4, Section 4.2.3:

\[
U(z)=U_{ref}\left(\frac{z}{z_{ref}}\right)^n
\]

ALOHA reports typical exponents for Pasquill classes A–F (for a reference roughness of 0.15 m); these are used as defaults and can be replaced with a site-calibrated `power_law_n_override`. An optional Monin–Obukhov profile is also available. It uses a stability/roughness-dependent Obukhov length and the Businger–Dyer momentum correction. The code normalizes either profile so that the input wind is recovered at the reference height.

For plume transport, the effective wind speed is estimated by vertically weighting `U(z)` with the reflected Gaussian vertical concentration profile. This is more representative than applying the 10 m wind speed uniformly to the whole plume. Note that power-law exponents taken from ALOHA's table are typical values, not a substitute for site meteorological measurements.

### Gaussian plume and puff

The steady plume retains the reflected-ground Gaussian solution. The finite release is represented by the finite-duration Gaussian cloud formulation: the longitudinal distribution is integrated analytically over each source-rate interval using error functions, while the lateral and vertical profiles are Gaussian with ground reflection. Constant-rate releases are integrated as one exact top-hat interval; variable-rate vessel blowdown is divided into piecewise-constant source-rate intervals. This is ALOHA-inspired and follows the finite-duration Gaussian-cloud concept documented in ALOHA Section 4.3, but it is not a byte-for-byte implementation of the proprietary ALOHA solver.

### Source term

- Direct rate: constant user-specified mass rate.
- Constant-pressure orifice: compressible ideal-gas orifice estimate for an initial rate.
- Vessel blowdown: rigid, adiabatic, well-mixed ideal-gas vessel. Mixture molecular weight and heat-capacity ratio are calculated from the selected gas composition. The blowdown approximation uses fixed composition and constant mixture Cp/gamma evaluated at 298.15 K; orifice flow is recalculated as vessel pressure and temperature fall.

### Computational grid

The `x` grid uses a fine linear spacing over the first 30 m by default and a geometrically stretched far field; `y` is clustered around the plume centreline; `z` is stretched toward the ground. This improves resolution near the source/ground without using an unnecessarily uniform high-resolution grid throughout the domain.

In [ ]:
# ALOHA Technical Documentation, Table 13 (Jones et al., 2013): Briggs dispersion coefficients.
STABILITY_ORDER = list("ABCDEF")
ALOHA_COEFFS = {
    "sx1": np.array([0.02, 0.02, 0.02, 0.04, 0.17, 0.17]),
    "sx2": np.array([1.22, 1.22, 1.22, 1.14, 0.97, 0.97]),
    "sy1": np.array([0.22, 0.16, 0.11, 0.08, 0.06, 0.04]),
    "sy2": np.array([0.0001] * 6),
    "sz1_rural": np.array([0.20, 0.12, 0.08, 0.06, 0.03, 0.016]),
    "sz2_rural": np.array([0.0, 0.0, 0.0002, 0.0015, 0.0003, 0.0003]),
    "sz3_rural": np.array([0.0, 0.0, -0.5, -0.5, -1.0, -1.0]),
    "sz1_urban": np.array([0.24, 0.24, 0.20, 0.14, 0.08, 0.08]),
    "sz2_urban": np.array([0.001, 0.001, 0.0, 0.0003, 0.0015, 0.0015]),
    "sz3_urban": np.array([0.5, 0.5, 0.0, -0.5, -0.5, -0.5]),
}

# Typical surface-layer power-law exponents documented in ALOHA for z0=0.15 m.
ALOHA_POWER_LAW_N = {"A": 0.109, "B": 0.112, "C": 0.120, "D": 0.142, "E": 0.203, "F": 0.253}


def dispersion_sigmas(distance_m, stability_class, terrain):
    """Return Briggs/ALOHA Table 13 sigma_x, sigma_y, sigma_z (m)."""
    if stability_class not in STABILITY_ORDER:
        raise ValueError("stability_class must be A, B, C, D, E, or F")
    if terrain not in {"Rural", "Urban"}:
        raise ValueError("terrain must be Rural or Urban")
    x = np.maximum(np.asarray(distance_m, dtype=float), 1.0)
    i = STABILITY_ORDER.index(stability_class)
    sx = ALOHA_COEFFS["sx1"][i] * x ** ALOHA_COEFFS["sx2"][i]
    sy = ALOHA_COEFFS["sy1"][i] * x / np.sqrt(1.0 + ALOHA_COEFFS["sy2"][i] * x)
    suffix = "rural" if terrain == "Rural" else "urban"
    sz = (ALOHA_COEFFS[f"sz1_{suffix}"][i] * x
          * (1.0 + ALOHA_COEFFS[f"sz2_{suffix}"][i] * x) ** ALOHA_COEFFS[f"sz3_{suffix}"][i])
    return sx, sy, sz


def obukhov_length_m(z0, stability_class):
    """Empirical stability-class/roughness relation documented in ALOHA section 4.2.3."""
    z0 = float(z0)
    if z0 <= 0:
        raise ValueError("Roughness length must be > 0.")
    relations = {
        "A": -11.4 * z0**0.10,
        "B": -26.0 * z0**0.17,
        "C": -123.0 * z0**0.30,
        "D": np.inf,
        "E": 123.0 * z0**0.30,
        "F": 26.0 * z0**0.17,
    }
    return float(relations[stability_class])


def psi_momentum(zeta):
    """Businger-Dyer momentum correction, conventional U=log-profile - psi_m form."""
    zeta = np.asarray(zeta, dtype=float)
    psi = np.zeros_like(zeta)
    unstable = zeta < 0.0
    stable = zeta > 0.0
    if np.any(unstable):
        a = np.power(np.maximum(1.0 - 16.0 * zeta[unstable], 1e-12), 0.25)
        psi[unstable] = (2.0 * np.log((1.0 + a) / 2.0)
                         + np.log((1.0 + a**2) / 2.0)
                         - 2.0 * np.arctan(a) + np.pi / 2.0)
    # Stable correction is negative in the U = log - psi convention.
    psi[stable] = -5.0 * zeta[stable]
    return psi


def wind_profile_parameters(met):
    """Return wind-profile parameters and friction-velocity estimate."""
    z0 = float(met["roughness_length_m"])
    zref = float(met["wind_reference_height_m"])
    uref = float(met["wind_speed_reference_m_s"])
    stability = met["stability_class"]
    model = met.get("wind_profile_model", "power_law")
    if z0 <= 0 or zref <= z0 or uref <= 0:
        raise ValueError("Require z0 > 0, reference height > z0, and positive reference wind speed.")
    if model not in {"power_law", "monin_obukhov"}:
        raise ValueError("wind_profile_model must be 'power_law' or 'monin_obukhov'.")
    L = obukhov_length_m(z0, stability)
    # Neutral log-law estimate is used for the Richardson screen and diagnostic only.
    u_star_neutral = KAPPA * uref / np.log((zref + z0) / z0)
    n_override = met.get("power_law_n_override", None)
    n = float(n_override) if n_override is not None else ALOHA_POWER_LAW_N[stability]
    if not (0.0 < n < 1.0):
        raise ValueError("Power-law exponent n must lie between 0 and 1.")
    # For the optional Monin-Obukhov profile, infer u* using the selected stability correction.
    zeta_ref = 0.0 if np.isinf(L) else zref / L
    psi_ref = float(psi_momentum(np.array([zeta_ref]))[0])
    denom = np.log((zref + z0) / z0) - psi_ref
    u_star_mo = KAPPA * uref / denom if denom > 0 else u_star_neutral
    return {"model": model, "z0_m": z0, "L_m": L,
            "u_star_m_s": u_star_neutral, "u_star_mo_m_s": u_star_mo,
            "power_law_n": n, "reference_height_m": zref,
            "reference_wind_m_s": uref, "stability_class": stability}


def wind_velocity(z_m, met):
    """Height-dependent wind speed U(z), normalized at the measured reference height."""
    pars = wind_profile_parameters(met)
    z = np.maximum(np.asarray(z_m, dtype=float), 0.0)
    zref, uref = pars["reference_height_m"], pars["reference_wind_m_s"]
    if pars["model"] == "power_law":
        return uref * (z / zref) ** pars["power_law_n"]
    L, z0, ustar = pars["L_m"], pars["z0_m"], pars["u_star_mo_m_s"]
    zeta = np.zeros_like(z) if np.isinf(L) else z / L
    psi = psi_momentum(zeta)
    profile = np.log((z + z0) / z0) - psi
    return (ustar / KAPPA) * np.maximum(profile, 0.0)


def effective_wind_speed(x_m, release_height_m, met):
    """Vertically concentration-weighted wind for transport at each x (m/s)."""
    x = np.atleast_1d(np.asarray(x_m, dtype=float))
    _, _, sz = dispersion_sigmas(x, met["stability_class"], met["terrain"])
    z_cap = max(250.0, float(release_height_m) + 8.0 * float(np.max(sz)))
    zq = np.linspace(0.0, z_cap, 360)
    uz = wind_velocity(zq, met)
    out = np.empty_like(x)
    for i, sigz in enumerate(sz):
        h = float(release_height_m)
        w = (np.exp(-0.5 * ((zq - h) / sigz)**2)
             + np.exp(-0.5 * ((zq + h) / sigz)**2))
        denominator = np.trapezoid(w, zq)
        out[i] = np.trapezoid(uz * w, zq) / max(denominator, 1e-15)
    return np.maximum(out, 0.05)


def make_stretched_domain(cfg):
    """Create nonuniform coordinates with exact source/receptor coordinates included.

    x=0 is the release plane. y_source and z_release are explicitly inserted so
    that a source-point concentration can be sampled without nearest-grid offsets.
    """
    d, src = cfg["domain"], cfg["source"]
    nx, ny, nz = int(d["nx"]), int(d["ny"]), int(d["nz"])
    xmax = float(d["x_max_m"])
    xref = min(float(d["x_refinement_end_m"]), xmax * 0.8)
    n_near = max(30, int(nx * 0.80))
    n_far = max(12, nx - n_near + 1)
    # Resolve the source/near field with a fine linear grid; stretch the far field geometrically.
    x_near = np.linspace(0.0, max(xref, 1.01), n_near)
    x_far = np.geomspace(max(xref, 1.01), xmax, n_far)
    x = np.unique(np.concatenate((x_near, x_far)))

    yhalf = float(d["y_half_width_m"])
    ysrc = float(src["source_y_m"])
    if abs(ysrc) > yhalf:
        raise ValueError("source.source_y_m must lie within +/- domain.y_half_width_m.")
    eta = np.linspace(-1.0, 1.0, ny)
    beta = float(d["y_stretch_factor"])
    y_base = yhalf * np.tanh(beta * eta) / np.tanh(beta)
    y = np.unique(np.concatenate((y_base, [ysrc, 0.0])))

    zmax = float(d["z_max_m"])
    zsrc = float(src["release_height_m"])
    zrec = float(d["receptor_height_m"])
    if zsrc > zmax:
        raise ValueError("domain.z_max_m must be at least source.release_height_m.")
    eta_z = np.linspace(0.0, 1.0, nz)
    z_base = zmax * eta_z ** float(d["z_stretch_power"])
    z = np.unique(np.concatenate((z_base, [min(zrec, zmax), zsrc, 0.0])))

    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
    return x, y, z, X, Y, Z


def mass_concentration_to_ppmv(c_kg_m3, mw_g_mol, temperature_K, pressure_Pa):
    mw_kg_mol = float(mw_g_mol) / 1000.0
    return (np.asarray(c_kg_m3) / mw_kg_mol) * R_UNIVERSAL * temperature_K / pressure_Pa * 1e6


def ideal_gas_density_kg_m3(mw_g_mol, temperature_K, pressure_Pa):
    return pressure_Pa * (mw_g_mol / 1000.0) / (R_UNIVERSAL * temperature_K)


def compressible_orifice_mass_flow(p_up, p_down, diameter_m, cd, gamma, temperature_K, mw_g_mol):
    """Single-phase, ideal-gas compressible orifice flow (kg/s)."""
    p_up, p_down = float(p_up), float(p_down)
    if p_up <= p_down:
        return 0.0, "no forward flow"
    if diameter_m <= 0 or not (0 < cd <= 1) or gamma <= 1 or temperature_K <= 0 or mw_g_mol <= 0:
        raise ValueError("Invalid orifice input(s).")
    area = np.pi * diameter_m**2 / 4.0
    rs = R_UNIVERSAL / (mw_g_mol / 1000.0)
    ratio = p_down / p_up
    critical = (2.0 / (gamma + 1.0)) ** (gamma / (gamma - 1.0))
    if ratio <= critical:
        q = (cd * area * p_up * np.sqrt(gamma / (rs * temperature_K))
             * (2.0 / (gamma + 1.0)) ** ((gamma + 1.0) / (2.0 * (gamma - 1.0))))
        return float(q), "choked"
    term = ratio ** (2.0 / gamma) - ratio ** ((gamma + 1.0) / gamma)
    q = cd * area * p_up * np.sqrt(2.0 * gamma * term / (rs * temperature_K * (gamma - 1.0)))
    return float(q), "unchoked"


def constant_pressure_orifice_rate(cfg):
    src, met, gas = cfg["source"], cfg["meteorology"], cfg["gas"]
    o = src["orifice"]
    return compressible_orifice_mass_flow(
        o["upstream_pressure_bar_abs"] * 1e5, met["pressure_Pa"],
        o["hole_diameter_mm"] / 1000.0, o["discharge_coefficient"],
        get_source_gamma(cfg, "orifice"), o["gas_temperature_K"], gas["molecular_weight_g_mol"])


def vessel_blowdown_profile(cfg):
    """Adiabatic, rigid, well-mixed ideal-gas vessel blowdown to near ambient pressure."""
    src, met, gas = cfg["source"], cfg["meteorology"], cfg["gas"]
    v = src["vessel"]
    p0 = float(v["initial_pressure_bar_abs"]) * 1e5
    t0 = float(v["initial_temperature_K"])
    volume = float(v["vessel_volume_m3"])
    gamma = get_source_gamma(cfg, "vessel")
    diameter = float(v["hole_diameter_mm"]) / 1000.0
    cd = float(v["discharge_coefficient"])
    pamb = float(met["pressure_Pa"])
    mw = float(gas["molecular_weight_g_mol"])
    rs = R_UNIVERSAL / (mw / 1000.0)
    if p0 <= pamb or volume <= 0 or t0 <= 0 or gamma <= 1:
        raise ValueError("Vessel blowdown requires P_initial > P_ambient, positive volume/T, and gamma > 1.")
    m0 = p0 * volume / (rs * t0)
    max_duration = float(v["maximum_release_duration_s"])
    output_dt = float(v["profile_output_step_s"])
    if max_duration <= 0 or output_dt <= 0:
        raise ValueError("Vessel maximum duration and output step must be positive.")

    def tank_state(m):
        frac = max(float(m) / m0, 1e-12)
        return p0 * frac**gamma, t0 * frac**(gamma - 1.0)

    def rhs(t, y):
        m = max(float(y[0]), 0.0)
        p, temp = tank_state(m)
        mdot, _ = compressible_orifice_mass_flow(p, pamb, diameter, cd, gamma, temp, mw)
        return [-mdot]

    def event_ambient(t, y):
        p, _ = tank_state(max(float(y[0]), 0.0))
        return p - pamb * 1.0001
    event_ambient.terminal = True
    event_ambient.direction = -1

    sol = solve_ivp(rhs, (0.0, max_duration), [m0], events=event_ambient,
                    dense_output=True, rtol=1e-7, atol=max(m0 * 1e-10, 1e-12),
                    max_step=max(output_dt / 2.0, 0.05))
    if not sol.success:
        raise RuntimeError("Vessel blowdown ODE failed: " + sol.message)
    if len(sol.t_events[0]):
        t_end = float(sol.t_events[0][0])
        end_reason = "pressure approached ambient"
    else:
        t_end = float(sol.t[-1])
        end_reason = "maximum configured release duration reached"
    times = np.arange(0.0, t_end, output_dt)
    if len(times) == 0 or times[-1] < t_end:
        times = np.append(times, t_end)
    masses = np.maximum(sol.sol(times)[0], 0.0)
    pressures = np.empty_like(times)
    temperatures = np.empty_like(times)
    mdots = np.empty_like(times)
    regimes = []
    for i, (tt, mm) in enumerate(zip(times, masses)):
        pressures[i], temperatures[i] = tank_state(mm)
        mdots[i], regime = compressible_orifice_mass_flow(
            pressures[i], pamb, diameter, cd, gamma, temperatures[i], mw)
        regimes.append(regime)
    mdots[pressures <= pamb * 1.0001] = 0.0
    profile = pd.DataFrame({"time_s": times, "tank_mass_kg": masses,
                            "tank_pressure_bar_abs": pressures / 1e5,
                            "tank_temperature_K": temperatures,
                            "mass_rate_kg_s": mdots, "flow_regime": regimes})
    metadata = {"initial_inventory_kg": m0, "remaining_mass_kg": float(masses[-1]),
                "released_mass_kg": float(m0 - masses[-1]), "end_reason": end_reason,
                "release_end_time_s": t_end}
    return profile, metadata


def get_source_setup(cfg):
    src = cfg["source"]
    mode = src["source_term_mode"]
    if mode == "direct_rate":
        q = float(src["mass_rate_kg_s"])
        if q <= 0:
            raise ValueError("mass_rate_kg_s must be positive")
        return {"kind": mode, "initial_rate_kg_s": q, "description": "User-specified direct gas release"}
    if mode == "orifice_initial":
        q, regime = constant_pressure_orifice_rate(cfg)
        return {"kind": mode, "initial_rate_kg_s": q,
                "description": f"Constant-pressure ideal-gas orifice initial estimate ({regime})"}
    if mode == "vessel_gas_blowdown":
        profile, meta = vessel_blowdown_profile(cfg)
        return {"kind": mode, "initial_rate_kg_s": float(profile.mass_rate_kg_s.iloc[0]),
                "profile": profile, "metadata": meta,
                "description": "Adiabatic ideal-gas vessel blowdown"}
    raise ValueError("Unknown source_term_mode")


def build_release_segments(cfg, source_setup):
    """Return source intervals [t_start,t_end] and their mean mass rates.

    Constant direct/orifice releases are represented as one exact top-hat interval.
    Time-varying vessel blowdown is represented by piecewise-constant rate segments,
    whose interval-integrated mass is calculated by trapezoidal integration.
    """
    src, transient = cfg["source"], cfg["puff"]
    if source_setup["kind"] in {"direct_rate", "orifice_initial"}:
        duration = float(src["release_duration_s"])
        q = float(source_setup["initial_rate_kg_s"])
        if duration <= 0 or q < 0:
            raise ValueError("Release duration must be positive and rate nonnegative.")
        starts = np.array([0.0])
        ends = np.array([duration])
        rates = np.array([q])
        total_mass = q * duration
        return starts, ends, rates, {
            "release_duration_s": duration,
            "total_released_mass_kg": total_mass,
            "n_source_segments": 1,
            "source_rate_approximation": "Exact finite-duration top-hat source",
        }

    profile = source_setup["profile"]
    t_end = float(profile.time_s.iloc[-1])
    dt = float(transient["release_interval_s"])
    if dt <= 0:
        raise ValueError("puff.release_interval_s must be positive for variable-rate blowdown.")
    edges = np.arange(0.0, t_end, dt)
    edges = np.append(edges, t_end)
    q_edges = np.interp(edges, profile.time_s.to_numpy(), profile.mass_rate_kg_s.to_numpy())
    masses = np.maximum(0.5 * (q_edges[:-1] + q_edges[1:]) * np.diff(edges), 0.0)
    rates = masses / np.maximum(np.diff(edges), 1e-15)
    starts, ends = edges[:-1], edges[1:]
    keep = rates > 0.0
    return starts[keep], ends[keep], rates[keep], {
        "release_duration_s": t_end,
        "total_released_mass_kg": float(np.sum(masses)),
        "n_source_segments": int(np.sum(keep)),
        "source_rate_approximation": f"Piecewise-constant blowdown history; dt={dt:g} s",
        **source_setup["metadata"],
    }


def transient_gaussian_release(X, Y, Z, cfg, segment_starts, segment_ends,
                               segment_rates, observation_time_s):
    """Finite-duration transient Gaussian cloud from a time-varying ground/elevated source.

    Each source interval is integrated analytically in the downwind direction using
    the error-function form of a short-duration Gaussian cloud (ALOHA technical
    documentation, Section 4.3). Lateral and vertical distributions are Gaussian,
    with ground reflection. The local effective wind and Briggs sigmas are evaluated
    at each downwind grid station. This is an ALOHA-inspired extension, not a byte-
    for-byte implementation of the proprietary ALOHA algorithm.
    """
    src, met = cfg["source"], cfg["meteorology"]
    H, ysrc = float(src["release_height_m"]), float(src["source_y_m"])
    t_obs = float(observation_time_s)
    x_axis = np.asarray(X[:, 0, 0], dtype=float)
    x_eval = np.maximum(x_axis, 0.0)
    sx, sy, sz = dispersion_sigmas(np.maximum(x_eval, 1.0), met["stability_class"], met["terrain"])
    ueff = effective_wind_speed(np.maximum(x_eval, 1.0), H, met)
    sx3, sy3, sz3 = sx[:, None, None], sy[:, None, None], sz[:, None, None]
    u3 = np.maximum(ueff[:, None, None], 0.05)

    lateral = np.exp(-0.5 * ((Y - ysrc) / sy3)**2) / (np.sqrt(2.0*np.pi) * sy3)
    vertical = (np.exp(-0.5 * ((Z - H) / sz3)**2)
                + np.exp(-0.5 * ((Z + H) / sz3)**2)) / (np.sqrt(2.0*np.pi) * sz3)

    C = np.zeros_like(X, dtype=float)
    sqrt2 = np.sqrt(2.0)
    used_segments = 0
    for t0, t1, qbar in zip(segment_starts, segment_ends, segment_rates):
        if qbar <= 0 or t_obs <= float(t0):
            continue
        # Clip the current segment at the observation time. If the source is active,
        # t1_eff == t_obs and the trailing edge remains connected to x=0.
        t1_eff = min(float(t1), t_obs)
        if t1_eff <= float(t0):
            continue
        age_latest = max(t_obs - t1_eff, 0.0)
        age_earliest = max(t_obs - float(t0), 0.0)
        arg_latest = (x_axis - ueff * age_latest) / (sqrt2 * sx)
        arg_earliest = (x_axis - ueff * age_earliest) / (sqrt2 * sx)
        longitudinal = 0.5 * (erf(arg_latest) - erf(arg_earliest))
        longitudinal = np.maximum(longitudinal, 0.0)[:, None, None]
        C += (float(qbar) / u3) * lateral * vertical * longitudinal
        used_segments += 1
    return np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0), used_segments


def continuous_gaussian_plume(X, Y, Z, cfg, source_setup):
    """Steady reflected-ground Gaussian plume including the x=0 source plane.

    The 1 m lower bound on Briggs sigmas regularizes the mathematical point source;
    it is not a resolved jet/nozzle near-field model.
    """
    src, met = cfg["source"], cfg["meteorology"]
    Q = float(source_setup["initial_rate_kg_s"])
    x_axis = np.maximum(np.asarray(X[:, 0, 0], dtype=float), 1.0)
    sx, sy, sz = dispersion_sigmas(x_axis, met["stability_class"], met["terrain"])
    ueff = effective_wind_speed(x_axis, src["release_height_m"], met)
    sy3, sz3, u3 = sy[:, None, None], sz[:, None, None], np.maximum(ueff[:, None, None], 0.05)
    H = float(src["release_height_m"])
    lateral = np.exp(-0.5 * ((Y - src["source_y_m"]) / sy3)**2)
    vertical = (np.exp(-0.5 * ((Z - H) / sz3)**2)
                + np.exp(-0.5 * ((Z + H) / sz3)**2))
    C = Q / (2.0 * np.pi * u3 * sy3 * sz3) * lateral * vertical
    return np.nan_to_num(np.where(X >= 0.0, C, 0.0), nan=0.0, posinf=0.0, neginf=0.0)


def transient_gaussian_receptor_concentration(point, time_s, cfg,
                                               segment_starts, segment_ends, segment_rates):
    """Direct transient finite-duration Gaussian concentration at a single receptor (kg/m3)."""
    xr, yr, zr = map(float, point)
    if xr < 0:
        return 0.0
    src, met = cfg["source"], cfg["meteorology"]
    H, ysrc = float(src["release_height_m"]), float(src["source_y_m"])
    x_eval = max(xr, 1.0)
    sx, sy, sz = [float(v[0]) for v in dispersion_sigmas(
        np.array([x_eval]), met["stability_class"], met["terrain"])]
    ueff = float(effective_wind_speed(np.array([x_eval]), H, met)[0])
    gy = np.exp(-0.5*((yr-ysrc)/sy)**2) / (np.sqrt(2*np.pi)*sy)
    gz = (np.exp(-0.5*((zr-H)/sz)**2) + np.exp(-0.5*((zr+H)/sz)**2)) / (np.sqrt(2*np.pi)*sz)
    total = 0.0
    t_obs = float(time_s)
    for t0, t1, qbar in zip(segment_starts, segment_ends, segment_rates):
        if qbar <= 0 or t_obs <= float(t0):
            continue
        t1_eff = min(float(t1), t_obs)
        if t1_eff <= float(t0):
            continue
        age_latest = max(t_obs-t1_eff, 0.0)
        age_earliest = max(t_obs-float(t0), 0.0)
        longitudinal = 0.5 * (
            erf((xr-ueff*age_latest)/(np.sqrt(2)*sx))
            - erf((xr-ueff*age_earliest)/(np.sqrt(2)*sx)))
        total += float(qbar)/max(ueff,0.05) * gy * gz * max(float(longitudinal),0.0)
    return float(max(total, 0.0))


def richardson_screening(cfg, source_setup):
    """ALOHA-style critical Richardson screen using initial source rate and inferred u*."""
    src, met, gas = cfg["source"], cfg["meteorology"], cfg["gas"]
    rho_g = ideal_gas_density_kg_m3(gas["molecular_weight_g_mol"], met["temperature_K"], met["pressure_Pa"])
    rho_a = ideal_gas_density_kg_m3(met["air_molecular_weight_g_mol"], met["temperature_K"], met["pressure_Pa"])
    wp = wind_profile_parameters(met)
    U10 = float(wind_velocity(10.0, met))
    Q = float(source_setup["initial_rate_kg_s"])
    Hstar = Q * np.pi / (4.0 * max(rho_g, 1e-12) * max(U10, 1e-6))
    g_reduced = G * (rho_g - rho_a) / max(rho_a, 1e-12)
    ri = Hstar * g_reduced / max(wp["u_star_m_s"]**2, 1e-12)
    return {"rho_gas_kg_m3": rho_g, "rho_air_kg_m3": rho_a,
            "u_star_m_s": wp["u_star_m_s"], "L_m": wp["L_m"],
            "U10_profile_m_s": U10, "source_characteristic_height_m": Hstar,
            "Ri_critical": ri,
            "screen": "Passive-Gaussian indication (Ri* < 1)" if ri < 1.0 else "Dense-gas effects may be important (Ri* >= 1)"}


def prepare_contour_display(values_ppm, plot_cfg):
    values = np.asarray(values_ppm, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0 or np.max(finite) <= 0:
        raise ValueError("This concentration slice is zero everywhere; check observation time, source rate, and domain.")
    maximum = float(np.max(finite))
    scale = plot_cfg.get("contour_scale", "log")
    if scale == "linear":
        return np.where(np.isfinite(values), values, 0.0), {
            "zmin": 0.0, "zmax": maximum,
            "colorbar": {"title": "Concentration (ppmv; linear)"}}, 0.0
    floor = min(float(plot_cfg.get("log_floor_ppm", 0.01)), maximum / 10.0)
    floor = max(floor, 1e-15)
    lo, hi = np.log10(floor), np.log10(maximum)
    display_values = np.log10(np.maximum(np.where(np.isfinite(values), values, 0.0), floor))
    ticks = np.unique(np.concatenate(([lo], np.arange(np.ceil(lo), np.floor(hi) + 1), [hi])))
    style = {"zmin": lo, "zmax": hi, "colorbar": {
        "title": "Concentration (ppmv; log)", "tickmode": "array",
        "tickvals": ticks.tolist(), "ticktext": [f"{10**v:.3g}" for v in ticks]}}
    return display_values, style, floor

print("Dispersion, meteorology, grid, source-term and plotting functions defined.")

### Finite-duration source integration and source-plane representation

For transient Gaussian calculations, the longitudinal concentration distribution is integrated over each active source interval using the Gaussian error function. A constant-rate release is represented as a finite-duration source interval; a time-varying vessel discharge is represented by piecewise-constant source-rate segments. This produces a continuous finite-duration cloud representation rather than a cloud made only from discrete midpoint emissions.

The computational grid includes the `x = 0` source plane and the exact source crosswind and release-height coordinates. While a release is active, concentration can therefore be present at the source plane. After the release stops, the cloud advects downwind and a source-to-cloud gap is physically expected. An isosurface may also appear detached if its selected ppmv display threshold exceeds the concentration at the source grid point. The notebook reports the source-plane concentration and threshold relationship. A 1 m minimum downwind evaluation distance regularises the point-source dispersion parameters; it does not resolve the nozzle jet or near-field momentum region.

## 4. RUN MODEL

This cell calculates the source history, refined domain, advection map, 3D concentration field, and receptor time histories. For `model_mode="puff"`, the solver calculates a finite-duration Gaussian cloud at `observation_time_s`; receptor traces show concentration over time. The default observation is 60 s, during the 120 s release, so the example snapshot should have a source-connected concentration region when the selected isosurface threshold is below the source-plane concentration.

In [ ]:
# Source term and finite-duration release history
SOURCE = get_source_setup(CONFIG)
if CONFIG["model_mode"] == "puff":
    SEGMENT_STARTS, SEGMENT_ENDS, SEGMENT_RATES, RELEASE_SUMMARY = build_release_segments(CONFIG, SOURCE)
else:
    SEGMENT_STARTS = SEGMENT_ENDS = SEGMENT_RATES = None
    RELEASE_SUMMARY = {"release_duration_s": np.inf, "total_released_mass_kg": np.inf}

# Refined nonuniform 3D computational grid. Exact source plane/location is included.
x, y, z, X, Y, Z = make_stretched_domain(CONFIG)
mode = CONFIG["model_mode"]
if mode == "continuous":
    C_kg_m3 = continuous_gaussian_plume(X, Y, Z, CONFIG, SOURCE)
    active_segments = None
else:
    C_kg_m3, active_segments = transient_gaussian_release(
        X, Y, Z, CONFIG, SEGMENT_STARTS, SEGMENT_ENDS, SEGMENT_RATES,
        CONFIG["puff"]["observation_time_s"])

met, gas, src = CONFIG["meteorology"], CONFIG["gas"], CONFIG["source"]
C_mg_m3 = C_kg_m3 * 1e6
C_ppm = mass_concentration_to_ppmv(C_kg_m3, gas["molecular_weight_g_mol"], met["temperature_K"], met["pressure_Pa"])
source_ijk = (int(np.argmin(abs(x-0.0))), int(np.argmin(abs(y-src["source_y_m"]))), int(np.argmin(abs(z-src["release_height_m"]))))
source_plane_ppm = float(C_ppm[source_ijk])
z_receptor = float(CONFIG["domain"]["receptor_height_m"])
z_index = int(np.argmin(np.abs(z - z_receptor)))
C_receptor_slice_ppm = C_ppm[:, :, z_index]
SCREEN = richardson_screening(CONFIG, SOURCE)

print(f"Mode: {mode}")
print(f"Source: {SOURCE['description']}")
print(f"Gas mixture: {gas['name']}")
print(f"Mixture composition basis: {CONFIG['mixture']['fraction_basis']}")
print(f"Initial discharge rate: {SOURCE['initial_rate_kg_s']:.5g} kg/s")
if mode == "puff":
    print(f"Total represented release mass: {RELEASE_SUMMARY['total_released_mass_kg']:.5g} kg")
    print(f"Source-rate segments: {len(SEGMENT_RATES):,}; contributing at snapshot: {active_segments}")
print(f"Grid: {len(x)} × {len(y)} × {len(z)} = {C_kg_m3.size:,} cells")
print(f"Source grid coordinate: x={x[0]:.3f} m; y-source included={np.any(np.isclose(y, src['source_y_m']))}; z-release included={np.any(np.isclose(z, src['release_height_m']))}")
source_ijk = (int(np.argmin(abs(x-0.0))), int(np.argmin(abs(y-src['source_y_m']))), int(np.argmin(abs(z-src['release_height_m']))))
print(f"Grid minimum spacings: dx={np.min(np.diff(x)):.3g} m, dy={np.min(np.diff(y)):.3g} m, dz={np.min(np.diff(z)):.3g} m")
print(f"Max concentration in snapshot: {np.max(C_ppm):,.4g} ppmv")
print(f"Concentration at exact source-plane grid point: {source_plane_ppm:,.4g} ppmv")
print(f"Richardson screen: Ri*={SCREEN['Ri_critical']:.4g} — {SCREEN['screen']}")

# Check that profile exactly recovers the reference observation and report stability parameters.
wind_par = wind_profile_parameters(met)
print(f"Wind-profile check: U({wind_par['reference_height_m']:.2f} m) = "
      f"{wind_velocity(wind_par['reference_height_m'], met):.4f} m/s "
      f"(input {wind_par['reference_wind_m_s']:.4f} m/s)")
print(f"Wind profile model: {wind_par['model']}; power-law n={wind_par['power_law_n']:.3f}; "
      f"neutral-log u* estimate={wind_par['u_star_m_s']:.4f} m/s; Obukhov L={wind_par['L_m']:.4g} m")
if wind_par["model"] == "power_law" and met.get("power_law_n_override") is None and not np.isclose(met["roughness_length_m"], 0.15):
    print("Note: default power-law n values are ALOHA's typical values for z0=0.15 m; consider a site-calibrated override if available.")

# Resolution diagnostics at the first downwind station and 10 m.
sx10, sy10, sz10 = dispersion_sigmas(np.array([10.0]), met["stability_class"], met["terrain"])
i10 = int(np.argmin(np.abs(x - 10.0)))
left_i, right_i = max(i10 - 1, 0), min(i10 + 1, len(x) - 1)
dx_local_10 = float((x[right_i] - x[left_i]) / max(right_i - left_i, 1))
resolution = pd.DataFrame([
    ["Minimum dx anywhere", np.min(np.diff(x)), "m"],
    ["Local dx near 10 m", dx_local_10, "m"],
    ["Minimum dy near centreline", np.min(np.diff(y)), "m"],
    ["Minimum dz near ground", np.min(np.diff(z)), "m"],
    ["sigma_x at 10 m", float(sx10[0]), "m"],
    ["sigma_y at 10 m", float(sy10[0]), "m"],
    ["sigma_z at 10 m", float(sz10[0]), "m"],
], columns=["Grid / plume diagnostic", "Value", "Unit"])
display(resolution)
if dx_local_10 > 0.5 * float(sx10[0]):
    warnings.warn("Local downwind spacing near 10 m is larger than half sigma_x; increase nx or move the x_refinement_end_m boundary farther downwind.")
if np.min(np.diff(y)) > 0.5 * float(sy10[0]):
    warnings.warn("Crosswind spacing near the centreline is wider than sigma_y at 10 m; refine ny or use a larger y_stretch_factor.")
if np.min(np.diff(z)) > 0.5 * float(sz10[0]):
    warnings.warn("Vertical spacing near ground is wider than sigma_z at 10 m; refine nz or increase z_stretch_power.")

summary_rows = [
    ["Model mode", mode], ["Source model", SOURCE["description"]],
    ["Gas mixture", gas["name"]], ["Molecular weight", f"{gas['molecular_weight_g_mol']:.4g} g/mol"],
    ["Mixture Cp at 298.15 K", f"{gas['cp_molar_j_mol_k']:.4g} J/(mol K)"],
    ["Mixture gamma", f"{gas['heat_capacity_ratio_gamma']:.5g}"],
    ["Specific gas constant", f"{gas['specific_gas_constant_j_kg_k']:.4g} J/(kg K)"],
    ["Gas ideal-gas density at ambient", f"{SCREEN['rho_gas_kg_m3']:.4g} kg/m³"],
    ["Wind at reference height", f"{met['wind_speed_reference_m_s']:.4g} m/s at {met['wind_reference_height_m']:.3g} m"],
    ["Stability / terrain", f"{met['stability_class']} / {met['terrain']}"],
    ["Maximum snapshot concentration", f"{np.max(C_ppm):.4g} ppmv"],
    ["Maximum at receptor height", f"{np.max(C_receptor_slice_ppm):.4g} ppmv"],
    ["Critical Richardson number", f"{SCREEN['Ri_critical']:.4g}"],
    ["Dense-gas applicability screen", SCREEN["screen"]],
]
if mode == "puff" and SOURCE["kind"] == "vessel_gas_blowdown":
    summary_rows.extend([
        ["Vessel initial inventory", f"{SOURCE['metadata']['initial_inventory_kg']:.4g} kg"],
        ["Vessel released mass", f"{SOURCE['metadata']['released_mass_kg']:.4g} kg"],
        ["Vessel release end", f"{SOURCE['metadata']['release_end_time_s']:.3g} s"],
        ["Vessel blowdown end condition", SOURCE['metadata']['end_reason']],
    ])
display(pd.DataFrame(summary_rows, columns=["Output", "Value"]))

if met["wind_speed_reference_m_s"] < 1.0:
    warnings.warn("Wind speed is below 1 m/s. Very low-wind / calm dispersion is outside this model's recommended use.")
if SCREEN["Ri_critical"] >= 1.0:
    warnings.warn("Dense-gas effects may be important. Gaussian concentrations/distances should not be treated as a heavy-gas prediction.")
if np.max(C_ppm) > 100000.0:
    warnings.warn("Peak gas mole fraction exceeds 10 vol% (100,000 ppmv). The passive/dilute Gaussian assumption may be questionable near the source; review density and source-jet effects.")

## 5. Atmospheric wind profile

The plot verifies that the specified reference wind speed is reproduced at its reference height. It shows the selected stability-dependent profile above the ground. Change `wind_profile_model` to compare ALOHA's typical power-law profile with the optional Monin–Obukhov profile.

In [ ]:
z_plot = np.linspace(0.0, min(CONFIG["domain"]["z_max_m"], 200.0), 250)
u_plot = wind_velocity(z_plot, CONFIG["meteorology"])
fig_wind = go.Figure()
fig_wind.add_trace(go.Scatter(x=u_plot, y=z_plot, mode="lines", name="U(z)"))
fig_wind.add_hline(y=CONFIG["meteorology"]["wind_reference_height_m"], line_dash="dash",
                   annotation_text="Wind reference height")
fig_wind.update_layout(title="Atmospheric wind-speed profile", xaxis_title="Wind speed U(z) (m/s)",
                       yaxis_title="Height above ground z (m)", height=520)
fig_wind.show()

## 6. Source release profile

For vessel blowdown, this graph displays tank pressure, temperature, and mass discharge rate over time. For direct-rate and constant-pressure-orifice scenarios, it displays the configured constant release history.

In [ ]:
if mode == "puff":
    if SOURCE["kind"] == "vessel_gas_blowdown":
        release_profile = SOURCE["profile"].copy()
    else:
        duration = float(src["release_duration_s"])
        tprof = np.linspace(0.0, duration, max(2, int(duration / max(CONFIG["puff"]["release_interval_s"], 1.0)) + 1))
        release_profile = pd.DataFrame({"time_s": tprof,
            "mass_rate_kg_s": np.full_like(tprof, SOURCE["initial_rate_kg_s"])})

    fig_src = go.Figure()
    fig_src.add_trace(go.Scatter(x=release_profile.time_s, y=release_profile.mass_rate_kg_s,
                                 mode="lines", name="Mass release rate"))
    fig_src.update_layout(title="Gas release-rate history", xaxis_title="Time (s)",
                          yaxis_title="Mass release rate (kg/s)", height=430)
    fig_src.show()

    if "tank_pressure_bar_abs" in release_profile.columns:
        fig_tank = go.Figure()
        fig_tank.add_trace(go.Scatter(x=release_profile.time_s, y=release_profile.tank_pressure_bar_abs,
                                      mode="lines", name="Tank pressure"))
        fig_tank.update_layout(title="Vessel pressure during blowdown", xaxis_title="Time (s)",
                               yaxis_title="Absolute pressure (bar)", height=400)
        fig_tank.show()
        fig_temp = go.Figure()
        fig_temp.add_trace(go.Scatter(x=release_profile.time_s, y=release_profile.tank_temperature_K,
                                      mode="lines", name="Tank temperature"))
        fig_temp.update_layout(title="Vessel temperature during adiabatic blowdown",
                               xaxis_title="Time (s)", yaxis_title="Temperature (K)", height=400)
        fig_temp.show()

    if CONFIG["export"]["write_release_profile_csv"]:
        release_profile.to_csv(CONFIG["export"]["release_profile_filename"], index=False)
        print("Saved", CONFIG["export"]["release_profile_filename"])
else:
    print("Continuous steady-release mode: no finite release history is defined.")

## 7. 3D concentration field at the selected observation time

The 3D isosurface threshold is a visualization level in ppmv, not a safety criterion. For very large grids, set `show_3d_isosurface=False` to avoid rendering delays in the browser.

In [ ]:
if CONFIG["plot"]["show_3d_isosurface"]:
    iso_level = float(CONFIG["plot"]["isosurface_level_ppm"])
    max_ppm = float(np.max(C_ppm))
    if max_ppm > 0:
        iso_plot = min(iso_level, max_ppm * 0.95)
        if iso_plot <= 0:
            iso_plot = max_ppm * 0.05
        fig3d = go.Figure(data=go.Isosurface(
            x=X.ravel(), y=Y.ravel(), z=Z.ravel(), value=C_ppm.ravel(),
            isomin=iso_plot, isomax=max_ppm, surface_count=4,
            opacity=0.22, caps=dict(x_show=False, y_show=False, z_show=False),
            colorbar=dict(title="Concentration (ppmv)"), name="Gas concentration"))
        fig3d.add_trace(go.Scatter3d(x=[0], y=[src["source_y_m"]], z=[src["release_height_m"]],
            mode="markers+text", marker=dict(size=6, color="red"), text=["Release source"],
            textposition="top center", name="Source"))
        fig3d.update_layout(title=f"3D gas concentration field — transient at t={CONFIG['puff']['observation_time_s']:.1f} s" if mode == "puff" else "3D steady Gaussian plume",
            scene=dict(xaxis_title="Downwind x (m)", yaxis_title="Crosswind y (m)",
                       zaxis_title="Height z (m)", aspectmode="manual",
                       aspectratio=dict(x=2.0, y=1.0, z=0.7)),
            margin=dict(l=0, r=0, t=55, b=0), height=700)
        fig3d.show()
        source_threshold = iso_plot
        if source_plane_ppm < source_threshold:
            print(f"Note: the displayed isosurface threshold ({source_threshold:.4g} ppmv) exceeds the concentration at the source-plane grid point ({source_plane_ppm:.4g} ppmv). A detached isosurface is therefore expected at this threshold; lower the isosurface threshold to visualize a source-connected contour if the field supports it.")
        elif mode == "puff" and CONFIG["puff"]["observation_time_s"] > RELEASE_SUMMARY["release_duration_s"]:
            print("The source has shut off before this snapshot. A plume separated from the source is physically expected.")
        else:
            print("The selected isosurface threshold is reached at the source-plane grid point; the concentration field is source-connected at the sampled source location.")
    else:
        print("No positive concentration values to plot.")
else:
    print("3D isosurface disabled in CONFIG['plot']['show_3d_isosurface'].")

## 8. Ground / receptor-height concentration contour

The default contour uses logarithmic colours to show the broad plume footprint. Hover text reports the actual, untransformed ppmv concentration. The selected receptor height is inserted in the nonuniform `z` grid.

In [ ]:
raw_ground = C_receptor_slice_ppm.T
C_ground_display, ground_style, ground_floor = prepare_contour_display(raw_ground, CONFIG["plot"])
fig_ground = go.Figure(data=go.Contour(
    x=x, y=y, z=C_ground_display, customdata=raw_ground,
    zmin=ground_style["zmin"], zmax=ground_style["zmax"], colorbar=ground_style["colorbar"],
    ncontours=28, contours=dict(coloring="fill", showlabels=False),
    hovertemplate="x: %{x:.2f} m<br>y: %{y:.2f} m<br>Concentration: %{customdata:.5g} ppmv<extra></extra>",
    name="Receptor-height concentration"))
fig_ground.add_trace(go.Scatter(x=[0], y=[src["source_y_m"]], mode="markers+text",
    marker=dict(color="red", size=10, symbol="star"), text=["Source"],
    textposition="top right", name="Source"))
scale_name = CONFIG["plot"]["contour_scale"]
fig_ground.update_layout(title=f"Concentration at z={z[z_index]:.2f} m ({scale_name} colour scale)",
    xaxis_title="Downwind distance x (m)", yaxis_title="Crosswind distance y (m)", height=600)
fig_ground.show()
print(f"Log colour floor (display only): {ground_floor:.4g} ppmv" if scale_name == "log" else "Linear colour scale selected.")
loc = float(CONFIG["plot"]["ground_contour_level_ppm"])
mask = C_receptor_slice_ppm >= loc
if np.any(mask):
    xx, yy = np.meshgrid(x, y, indexing="ij")
    print(f"{loc:g} ppmv is exceeded as far as x={np.max(xx[mask]):.1f} m and |y|={np.max(np.abs(yy[mask])):.1f} m within this domain.")
else:
    print(f"{loc:g} ppmv is not exceeded at the selected receptor height in this domain.")

## 9. Vertical cross-section through plume centreline

In [ ]:
j_center = int(np.argmin(np.abs(y - src["source_y_m"])))
raw_xz = C_ppm[:, j_center, :].T
xz_display, xz_style, xz_floor = prepare_contour_display(raw_xz, CONFIG["plot"])
fig_xz = go.Figure(data=go.Contour(
    x=x, y=z, z=xz_display, customdata=raw_xz,
    zmin=xz_style["zmin"], zmax=xz_style["zmax"], colorbar=xz_style["colorbar"],
    ncontours=28, contours=dict(coloring="fill", showlabels=False),
    hovertemplate="x: %{x:.2f} m<br>z: %{y:.2f} m<br>Concentration: %{customdata:.5g} ppmv<extra></extra>",
    name="Centreline cross-section"))
fig_xz.add_hline(y=src["release_height_m"], line_dash="dash", annotation_text="Release height")
fig_xz.update_layout(title=f"Vertical centreline cross-section ({scale_name} colour scale)",
    xaxis_title="Downwind distance x (m)", yaxis_title="Height z (m)", height=550)
fig_xz.show()

## 10. Time histories at receptors

For transient mode, concentration at each receptor is evaluated directly from the finite-duration Gaussian source-interval solution at each output time (rather than reading from a coarse interpolated 3D array). This gives a time history for arrival, peak and decay at user-defined receptor locations.

In [ ]:
if mode == "puff":
    history_end = float(CONFIG["puff"]["history_end_s"])
    history_dt = float(CONFIG["puff"]["history_step_s"])
    if history_dt <= 0 or history_end <= 0:
        raise ValueError("history_end_s and history_step_s must be positive.")
    t_history = np.arange(0.0, history_end + 0.5 * history_dt, history_dt)
    history_rows = []
    RECEPTOR_HISTORY_PPMV = {}
    fig_hist = go.Figure()
    for receptor in RECEPTORS:
        p = (receptor["x_m"], receptor["y_m"], receptor["z_m"])
        ckg = np.array([transient_gaussian_receptor_concentration(
            p, tt, CONFIG, SEGMENT_STARTS, SEGMENT_ENDS, SEGMENT_RATES) for tt in t_history])
        cppm = mass_concentration_to_ppmv(ckg, gas["molecular_weight_g_mol"],
                                          met["temperature_K"], met["pressure_Pa"])
        RECEPTOR_HISTORY_PPMV[receptor["name"]] = np.asarray(cppm, dtype=float)
        fig_hist.add_trace(go.Scatter(x=t_history, y=cppm, mode="lines", name=receptor["name"]))
        if np.max(cppm) > 0:
            imax = int(np.argmax(cppm))
            arrival_idx = np.where(cppm > max(np.max(cppm) * 0.01, 1e-12))[0]
            first_arrival = float(t_history[arrival_idx[0]]) if len(arrival_idx) else np.nan
        else:
            imax, first_arrival = 0, np.nan
        history_rows.append({**receptor, "peak_concentration_ppmv": float(np.max(cppm)),
                             "time_of_peak_s": float(t_history[imax]),
                             "first_1pct_of_peak_s": first_arrival})
    fig_hist.update_layout(title="Time-dependent finite-release concentration at receptors",
        xaxis_title="Time after release start (s)", yaxis_title="Concentration (ppmv)",
        yaxis_type="log", height=520)
    fig_hist.show()
    receptor_history_summary = pd.DataFrame(history_rows)
    display(receptor_history_summary)
else:
    print("Steady continuous plume mode has no finite-release arrival/decay history. Set model_mode='puff' to calculate transient receptor histories.")

## 11. Receptor snapshot values and optional CSV exports

In [ ]:
# Compare direct receptor evaluation with interpolation from the 3D grid.
from scipy.interpolate import RegularGridInterpolator
interp_snapshot = RegularGridInterpolator((x, y, z), C_kg_m3, bounds_error=False, fill_value=np.nan)
receptor_snapshot = []
for receptor in RECEPTORS:
    point = (receptor["x_m"], receptor["y_m"], receptor["z_m"])
    if mode == "puff":
        ckg_direct = transient_gaussian_receptor_concentration(
            point, CONFIG["puff"]["observation_time_s"], CONFIG,
            SEGMENT_STARTS, SEGMENT_ENDS, SEGMENT_RATES)
    else:
        xr, yr, zr = point
        sxr, syr, szr = dispersion_sigmas(max(xr, 1.0), met["stability_class"], met["terrain"])
        ur = float(effective_wind_speed(np.array([max(xr, 1.0)]), src["release_height_m"], met)[0])
        h = float(src["release_height_m"])
        vert = np.exp(-0.5*((zr-h)/szr)**2) + np.exp(-0.5*((zr+h)/szr)**2)
        lat = np.exp(-0.5*((yr-src["source_y_m"])/syr)**2)
        ckg_direct = float(SOURCE["initial_rate_kg_s"] / (2*np.pi*ur*syr*szr) * lat * vert) if xr >= 0 else 0.0
    ckg_interp = float(interp_snapshot([point])[0])
    cppm_direct = float(mass_concentration_to_ppmv(ckg_direct, gas["molecular_weight_g_mol"], met["temperature_K"], met["pressure_Pa"]))
    cppm_interp = float(mass_concentration_to_ppmv(ckg_interp, gas["molecular_weight_g_mol"], met["temperature_K"], met["pressure_Pa"])) if np.isfinite(ckg_interp) else np.nan
    # Relative interpolation error is not informative for concentrations numerically close to zero.
    interpolation_threshold_ppm = max(1e-6, 1e-4 * float(np.max(C_ppm)))
    relerr = (100.0 * abs(ckg_interp - ckg_direct) / max(abs(ckg_direct), 1e-15)
              if np.isfinite(ckg_interp) and cppm_direct >= interpolation_threshold_ppm else np.nan)
    receptor_snapshot.append({**receptor, "direct_snapshot_ppmv": cppm_direct,
                              "grid_interpolated_ppmv": cppm_interp,
                              "grid_interpolation_error_pct": relerr,
                              "interpolation_check_threshold_ppmv": interpolation_threshold_ppm})
receptor_snapshot_df = pd.DataFrame(receptor_snapshot)
display(receptor_snapshot_df)
if receptor_snapshot_df.grid_interpolation_error_pct.notna().any():
    worst_err = receptor_snapshot_df.grid_interpolation_error_pct.max()
    print(f"Worst receptor grid-interpolation difference above {interpolation_threshold_ppm:.3g} ppmv: {worst_err:.2f}%")
    if worst_err > 10.0:
        warnings.warn("Grid interpolation error at one or more receptors exceeds 10%; refine the nonuniform grid or place receptors on/near grid nodes.")

if CONFIG["export"]["write_full_3d_csv"]:
    wind_to = np.deg2rad((met["wind_direction_from_deg"] + 180.0) % 360.0)
    east = X * np.sin(wind_to) + Y * np.cos(wind_to)
    north = X * np.cos(wind_to) - Y * np.sin(wind_to)
    field_df = pd.DataFrame({"x_downwind_m": X.ravel(), "y_crosswind_m": Y.ravel(),
        "z_m": Z.ravel(), "east_offset_m": east.ravel(), "north_offset_m": north.ravel(),
        "mixture_concentration_kg_m3": C_kg_m3.ravel(), "mixture_concentration_mg_m3": C_mg_m3.ravel(),
        "mixture_concentration_ppmv": C_ppm.ravel()})
    # With fixed gas composition, each component concentration is the corresponding
    # mixture mass/mole fraction multiplied by the total mixture concentration.
    for mix_row in gas["composition_table"].itertuples(index=False):
        formula = str(getattr(mix_row, "Formula")).replace("/", "_").replace(" ", "")
        mole_fraction = float(getattr(mix_row, "_4")) / 100.0  # column 5: Mole fraction (%)
        mass_fraction = float(getattr(mix_row, "_5")) / 100.0  # column 6: Mass fraction (%)
        field_df[f"{formula}_mass_concentration_kg_m3"] = C_kg_m3.ravel() * mass_fraction
        field_df[f"{formula}_concentration_ppmv"] = C_ppm.ravel() * mole_fraction
    field_df.to_csv(CONFIG["export"]["csv_filename"], index=False)
    print(f"Saved {CONFIG['export']['csv_filename']} ({len(field_df):,} grid cells).")
else:
    print("3D CSV export disabled. Set CONFIG['export']['write_full_3d_csv']=True to export the snapshot field.")

if mode == "puff" and SOURCE["kind"] == "vessel_gas_blowdown" and CONFIG["export"]["write_release_profile_csv"]:
    SOURCE["profile"].to_csv(CONFIG["export"]["release_profile_filename"], index=False)
    print(f"Saved {CONFIG['export']['release_profile_filename']}.")

## 13. Component-specific flammability, diffusivity and toxicity assessment

This section adds component-specific LFL/UFL data, Fuller-Schettler-Giddings estimates of binary gas-in-air molecular diffusivity, and available acute toxicity endpoints. For mixed fuels, the Le Chatelier rule estimates a blend LFL/UFL from the combustible components. Inert components are excluded from the blend's fuel composition but remain part of the total release mixture.

**Interpretation:** the LFL/UFL figures are reference limits in air, not a combustion simulation. The toxicity contours in the 2D plots are instantaneous concentration overlays at the displayed snapshot time; they are not duration-averaged AEGL/ERPG/IDLH threat zones. Receptor screening separately calculates endpoint-window average concentration only if the simulated history is long enough. Blank database fields mean unavailable/not entered, never zero.


In [ ]:
# Component-resolved hazard fields from the mixture snapshot.
hazard_cfg = CONFIG["hazard_assessment"]
hazard_props = CONFIG["hazard_properties"]
component_y = gas["composition_mole_fractions"]
selected_endpoint = hazard_cfg["toxicity_endpoint"]

# Concentration field at receptor height and centreline, in component ppmv.
component_ground_ppm = {name: raw_ground * float(yi) for name, yi in component_y.items()}
component_xz_ppm = {name: raw_xz * float(yi) for name, yi in component_y.items()}

# Mixed-fuel concentration is total gas-mixture concentration times total fuel mole fraction.
fuel_y = hazard_props["fuel_mole_fraction_in_released_mixture"]
fuel_ground_ppm = raw_ground * fuel_y
fuel_xz_ppm = raw_xz * fuel_y
lfl_mix_ppm = hazard_props["mixed_fuel_lfl_ppm"]
ufl_mix_ppm = hazard_props["mixed_fuel_ufl_ppm"]


def add_isoline(fig, xvals, yvals, field, threshold, label, line_color):
    """Add one isoline where the current plotted field intersects threshold."""
    if threshold is None or not np.isfinite(threshold) or field.size == 0:
        return False
    vmin = float(np.nanmin(field))
    vmax = float(np.nanmax(field))
    if not (vmin <= float(threshold) <= vmax):
        return False
    eps = max(abs(float(threshold)) * 1e-6, 1e-8)
    fig.add_trace(go.Contour(
        x=xvals, y=yvals, z=field,
        autocontour=False, showscale=False, showlegend=True, name=label,
        contours=dict(start=float(threshold), end=float(threshold) + eps,
                      size=eps, coloring="lines", showlabels=True,
                      labelfont=dict(size=10)),
        line=dict(color=line_color, width=2),
        hoverinfo="skip",
    ))
    return True


hazard_summary_rows = []
flammable_names = []
toxic_names = []
for name, yi in component_y.items():
    rec = CHEMICAL_DATABASE[name]
    endpoint_value, endpoint_window_s, endpoint_label, endpoint_note = selected_toxicity_endpoint(rec, selected_endpoint)
    Dair = fuller_diffusivity_in_air_m2_s(
        rec, met["temperature_K"], met["pressure_Pa"],
        met.get("air_molecular_weight_g_mol", FULLER_AIR_MW_G_MOL)
    )
    if rec.get("lfl_vol_pct") is not None and rec.get("ufl_vol_pct") is not None:
        flammable_names.append(name)
    if endpoint_value is not None:
        toxic_names.append(name)
    hazard_summary_rows.append({
        "Chemical": name,
        "Mole fraction in released gas (%)": 100.0 * float(yi),
        "Component snapshot max (ppmv)": float(np.nanmax(C_ppm) * float(yi)),
        "LFL (vol% in air)": rec.get("lfl_vol_pct"),
        "UFL (vol% in air)": rec.get("ufl_vol_pct"),
        "D_i-air (m2/s)": Dair,
        "Selected endpoint": endpoint_label,
        "Endpoint (ppm)": endpoint_value,
        "Endpoint window (s)": endpoint_window_s,
        "IDLH basis": rec.get("idlh_basis"),
        "Data note": rec.get("hazard_note"),
    })
hazard_summary_df = pd.DataFrame(hazard_summary_rows)
print(f"Selected toxicity endpoint family: {selected_endpoint}")
print("Component hazard summary:")
display(hazard_summary_df)

# Mixed fuel LFL/UFL contours at receptor height and through the plume centreline.
if hazard_cfg["plot_hazard_contours"] and lfl_mix_ppm is not None:
    print(f"Le Chatelier estimated mixed-fuel LFL = {hazard_props['mixed_fuel_lfl_vol_pct']:.4g} vol% "
          f"({lfl_mix_ppm:.5g} ppm fuel vapour in air); UFL = {hazard_props['mixed_fuel_ufl_vol_pct']:.4g} vol% "
          f"({ufl_mix_ppm:.5g} ppm). Combustible components: {', '.join(flammable_names)}")
    fig_fl_ground = go.Figure(go.Contour(
        x=x, y=y, z=np.log10(np.maximum(fuel_ground_ppm, 1e-12)),
        colorscale="Viridis", colorbar=dict(title="log10 fuel ppmv"),
        contours=dict(coloring="fill", showlabels=False),
        hovertemplate="x=%{x:.1f} m<br>y=%{y:.1f} m<br>log10 fuel ppm=%{z:.3f}<extra></extra>",
        name="Fuel-vapour concentration"))
    lfl_drawn = add_isoline(fig_fl_ground, x, y, fuel_ground_ppm, lfl_mix_ppm,
                            f"Mixed LFL {lfl_mix_ppm:.0f} ppm", "red")
    ufl_drawn = add_isoline(fig_fl_ground, x, y, fuel_ground_ppm, ufl_mix_ppm,
                            f"Mixed UFL {ufl_mix_ppm:.0f} ppm", "orange")
    fig_fl_ground.update_layout(title="Mixed-fuel flammability screening — ground/receptor-height snapshot",
        xaxis_title="Downwind distance x (m)", yaxis_title="Crosswind distance y (m)", height=600)
    fig_fl_ground.show()

    # Individual-component pure-gas-in-air LFL/UFL reference lines. For multicomponent
    # fuel clouds, the Le Chatelier mixed-fuel contour above is the preferred blend screen.
    fig_component_fl = go.Figure(go.Contour(
        x=x, y=y, z=C_ground_display, zmin=ground_style["zmin"], zmax=ground_style["zmax"],
        colorscale="Viridis", colorbar=ground_style["colorbar"],
        contours=dict(coloring="fill", showlabels=False), name="Mixture concentration"))
    component_reference_lines = []
    comp_colors = ["red", "orange", "magenta", "cyan", "white", "yellow"]
    for k, name in enumerate(flammable_names):
        rec = CHEMICAL_DATABASE[name]
        yi = float(component_y[name])
        comp_ground = component_ground_ppm[name]
        comp_color = comp_colors[k % len(comp_colors)]
        lfl_ref = float(rec["lfl_vol_pct"]) * 10000.0
        ufl_ref = float(rec["ufl_vol_pct"]) * 10000.0
        if add_isoline(fig_component_fl, x, y, comp_ground, lfl_ref,
                       f"{name} pure-component LFL ref {lfl_ref:.0f} ppm", comp_color):
            component_reference_lines.append(f"{name} LFL")
        if add_isoline(fig_component_fl, x, y, comp_ground, ufl_ref,
                       f"{name} pure-component UFL ref {ufl_ref:.0f} ppm", comp_color):
            component_reference_lines.append(f"{name} UFL")
    fig_component_fl.update_layout(
        title="Individual-component pure-gas LFL/UFL reference overlays — ground snapshot",
        xaxis_title="Downwind distance x (m)", yaxis_title="Crosswind distance y (m)", height=620)
    fig_component_fl.show()
    print("Individual-component reference lines intersecting this snapshot:", component_reference_lines or "none")

    fig_fl_xz = go.Figure(go.Contour(
        x=x, y=z, z=np.log10(np.maximum(fuel_xz_ppm, 1e-12)),
        colorscale="Viridis", colorbar=dict(title="log10 fuel ppmv"),
        contours=dict(coloring="fill", showlabels=False), name="Fuel-vapour concentration"))
    add_isoline(fig_fl_xz, x, z, fuel_xz_ppm, lfl_mix_ppm,
                f"Mixed LFL {lfl_mix_ppm:.0f} ppm", "red")
    add_isoline(fig_fl_xz, x, z, fuel_xz_ppm, ufl_mix_ppm,
                f"Mixed UFL {ufl_mix_ppm:.0f} ppm", "orange")
    fig_fl_xz.update_layout(title="Mixed-fuel flammability screening — vertical centreline snapshot",
        xaxis_title="Downwind distance x (m)", yaxis_title="Height z (m)", height=560)
    fig_fl_xz.show()
    if not lfl_drawn:
        print("The current ground-level snapshot does not reach the estimated mixed-fuel LFL; no LFL isoline is expected.")
    if not ufl_drawn:
        print("The current ground-level snapshot does not reach the estimated mixed-fuel UFL; no UFL isoline is expected.")
else:
    print("Mixed-fuel LFL/UFL contours are not generated: the selected mixture has no complete numeric LFL/UFL blend data.")

# Toxicity endpoint contours are component-specific; no mixture-additivity assumption is applied.
if hazard_cfg["plot_hazard_contours"] and toxic_names:
    fig_tox_ground = go.Figure(go.Contour(
        x=x, y=y, z=C_ground_display, zmin=ground_style["zmin"], zmax=ground_style["zmax"],
        colorscale="Viridis", colorbar=ground_style["colorbar"],
        contours=dict(coloring="fill", showlabels=False), name="Total mixture concentration"))
    tox_ground_drawn = []
    color_cycle = ["red", "orange", "magenta", "cyan", "white", "yellow"]
    for k, name in enumerate(toxic_names):
        rec = CHEMICAL_DATABASE[name]
        threshold, _, label, _ = selected_toxicity_endpoint(rec, selected_endpoint)
        did = add_isoline(fig_tox_ground, x, y, component_ground_ppm[name], threshold,
                          f"{name}: {label} {threshold:g} ppm", color_cycle[k % len(color_cycle)])
        if did:
            tox_ground_drawn.append(name)
    fig_tox_ground.update_layout(title=f"Component-specific {selected_endpoint} concentration overlays — ground/receptor-height snapshot",
        xaxis_title="Downwind distance x (m)", yaxis_title="Crosswind distance y (m)", height=620)
    fig_tox_ground.show()

    fig_tox_xz = go.Figure(go.Contour(
        x=x, y=z, z=np.log10(np.maximum(raw_xz, 1e-12)),
        colorscale="Viridis", colorbar=dict(title="log10 total mixture ppmv"),
        contours=dict(coloring="fill", showlabels=False), name="Total mixture concentration"))
    tox_xz_drawn = []
    for k, name in enumerate(toxic_names):
        rec = CHEMICAL_DATABASE[name]
        threshold, _, label, _ = selected_toxicity_endpoint(rec, selected_endpoint)
        did = add_isoline(fig_tox_xz, x, z, component_xz_ppm[name], threshold,
                          f"{name}: {label} {threshold:g} ppm", color_cycle[k % len(color_cycle)])
        if did:
            tox_xz_drawn.append(name)
    fig_tox_xz.update_layout(title=f"Component-specific {selected_endpoint} concentration overlays — vertical centreline snapshot",
        xaxis_title="Downwind distance x (m)", yaxis_title="Height z (m)", height=560)
    fig_tox_xz.show()
    print("Toxic endpoint contours intersecting the displayed ground snapshot:", tox_ground_drawn or "none")
    print("Toxic endpoint contours intersecting the displayed vertical snapshot:", tox_xz_drawn or "none")
else:
    print(f"No selected component has a published/entered {selected_endpoint} endpoint; toxic endpoint contours are not generated.")

# Duration-aware receptor screening: only evaluate a duration endpoint if the simulated
# concentration history spans the full endpoint window. This avoids equating a snapshot
# concentration with an AEGL/ERPG/IDLH exposure assessment.
exposure_rows = []
if mode == "puff" and isinstance(RECEPTOR_HISTORY_PPMV, dict) and len(RECEPTOR_HISTORY_PPMV):
    dt_hist = float(CONFIG["puff"]["history_step_s"])
    t_hist = np.asarray(t_history, dtype=float)
    for receptor in RECEPTORS:
        mix_hist = np.asarray(RECEPTOR_HISTORY_PPMV.get(receptor["name"], []), dtype=float)
        for name, yi in component_y.items():
            rec = CHEMICAL_DATABASE[name]
            threshold, window_s, endpoint_label, endpoint_note = selected_toxicity_endpoint(rec, selected_endpoint)
            if threshold is None:
                continue
            if len(mix_hist) < 2 or t_hist[-1] - t_hist[0] + 1e-9 < window_s:
                exposure_rows.append({
                    "Receptor": receptor["name"], "Chemical": name, "Endpoint": endpoint_label,
                    "Threshold (ppm)": threshold, "Required averaging window (s)": window_s,
                    "Maximum window-average (ppm)": np.nan, "Peak window end time (s)": np.nan,
                    "Assessment": f"NOT ASSESSED: history shorter than {window_s:.0f} s",
                })
                continue
            nwin = int(round(window_s / dt_hist)) + 1
            if nwin > len(mix_hist):
                assessment = "NOT ASSESSED: insufficient samples for averaging window"
                peak_avg = peak_end = np.nan
            else:
                comp_hist = mix_hist * float(yi)
                kernel = np.ones(nwin, dtype=float) / nwin
                moving_avg = np.convolve(comp_hist, kernel, mode="valid")
                idx = int(np.argmax(moving_avg))
                peak_avg = float(moving_avg[idx])
                peak_end = float(t_hist[idx + nwin - 1])
                assessment = "EXCEEDS selected endpoint" if peak_avg >= threshold else "Below selected endpoint"
            exposure_rows.append({
                "Receptor": receptor["name"], "Chemical": name, "Endpoint": endpoint_label,
                "Threshold (ppm)": threshold, "Required averaging window (s)": window_s,
                "Maximum window-average (ppm)": peak_avg, "Peak window end time (s)": peak_end,
                "Assessment": assessment,
            })
else:
    print("Duration-aware receptor endpoint screening requires model_mode='puff' and a receptor time history.")

if exposure_rows:
    exposure_screen_df = pd.DataFrame(exposure_rows)
    display(exposure_screen_df)
    if any(str(x).startswith("NOT ASSESSED") for x in exposure_screen_df["Assessment"]):
        print("Increase CONFIG['puff']['history_end_s'] to at least the selected endpoint window (typically 3600 s for AEGL-2/ERPG-2 or 1800 s for IDLH) to perform duration-aware receptor screening.")

# Export the component hazard properties as a portable CSV if desired.
if CONFIG["export"].get("write_hazard_database_csv", False):
    hazard_table_for_display.to_csv(CONFIG["export"].get("hazard_database_filename", "selected_component_hazards.csv"), index=False)
    print("Saved selected_component_hazards.csv or configured hazard database filename.")


## 14. Optional dense-gas dispersion — DEGADISx

This section is only executed when `CONFIG["model_engine"] == "degadisx_ground"`. It uses the external `degadisx` Python package (version 0.1.4) to run the DEGADIS 2.1 ground-level steady or transient calculation.

**Source applicability is strict:** the integrated route is for a ground-level, low-momentum area source over flat, level terrain. The source rate and radius must describe the gas source *after* any upstream release/flash/jet/rainout analysis. It is not valid to pass the raw discharge from the existing vessel/orifice calculation directly into this ground-area deck. Use a separate jet-to-ground bridge or an independently resolved ground-source mass-rate history for those cases.

DEGADISx reports a downwind profile. The 3D plots below reconstruct a visualization field from its centreline contaminant concentration, core half-width `b`, lateral scale `sy`, vertical scale `sz`, and the model's vertical exponent. The underlying solver remains DEGADISx; this Cartesian array is a post-processing approximation for visualization, not a native 3D DEGADIS solution or CFD result.


In [ ]:
# Optional DEGADISx dense-gas engine.
# Installed only when this engine is selected; requires an internet-enabled Colab runtime.
import importlib
import subprocess
import sys
import tempfile
from pathlib import Path

DEGADISX_RESULT = None
DEGADISX_PROFILE = None
DEGADISX_SOURCE_RESULT = None
DEGADISX_SNAPSHOT = None
DEGADISX_FIELD_PPM = None
DEGADISX_FIELD_MOLE = None
DEGADISX_X = DEGADISX_Y = DEGADISX_Z = None
DEGADISX_HISTORY_DF = None


def _degadisx_imports():
    """Import pinned DEGADISx release, installing it in Colab if absent."""
    try:
        from degadisx.run import run_steady, run_transient, Receptor
    except ImportError:
        print("Installing DEGADISx 0.1.4 (requires internet access in this runtime)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "degadisx==0.1.4"])
        importlib.invalidate_caches()
        from degadisx.run import run_steady, run_transient, Receptor
    return run_steady, run_transient, Receptor


def _degadisx_flammability_limits(cfg):
    """Return contaminant-cloud mole-fraction limits for the pseudo-component.

    The database's Le Chatelier blend LFL/UFL is adjusted for the fuel mole
    fraction in the released pseudo-component. If the selected released mixture
    contains no fuel, a numerical reporting window is supplied; it is NOT a
    statement that the inert mixture is flammable.
    """
    hp = cfg["hazard_properties"]
    fuel_y = float(hp.get("fuel_mole_fraction_in_released_mixture", 0.0) or 0.0)
    lfl = hp.get("mixed_fuel_lfl_vol_pct")
    ufl = hp.get("mixed_fuel_ufl_vol_pct")
    cutoff = float(cfg["heavy_gas"]["mole_fraction_cutoff"])
    if fuel_y > 0.0 and (lfl is None or ufl is None):
        # DEGADISx requires numerical reporting bounds in its input deck. These
        # placeholders are not a physical flammability envelope and are not used
        # to draw hazard contours when mixture LFL/UFL are unavailable.
        return 1.0, cutoff, (
            "Fuel/pyrophoric component is present but numeric mixed LFL and/or UFL "
            "is unavailable. DEGADISx deck bounds are numerical placeholders only; "
            "do not interpret them as flammability limits."
        )
    if fuel_y > 0.0 and lfl is not None and ufl is not None:
        llc = (float(lfl) / 100.0) / fuel_y
        ulc = (float(ufl) / 100.0) / fuel_y
        if llc >= 1.0 or ulc >= 1.0:
            return 1.0, 1.0, "The released blend is too fuel-dilute to reach its estimated mixed LFL at any gas-in-air fraction <= 1."
        llc = max(1.0e-12, min(llc, 1.0))
        ulc = max(llc, min(ulc, 1.0))
        return ulc, llc, "Mixed-fuel Le Chatelier limits, adjusted to total released-mixture mole fraction."
    return 1.0, cutoff, "No fuel component in selected mixture; lower level is a numerical reporting cutoff, not a flammability limit."


def _write_degadisx_inp(deck_path, cfg):
    """Write a DEGADISx ground-level .INP deck using its documented field order."""
    met = cfg["meteorology"]
    hg = cfg["heavy_gas"]
    src = cfg["source"]
    gas = cfg["gas"]
    if cfg["model_engine"] != "degadisx_ground":
        raise ValueError("DEGADISx deck requested while model_engine is not degadisx_ground.")
    if src["source_term_mode"] != "direct_rate":
        raise ValueError("Ground-area DEGADISx integration requires a resolved direct mass-rate source.")
    if float(src["release_height_m"]) > 1.0e-6:
        raise ValueError("Ground-area DEGADISx route requires release height = 0 m.")
    if float(src.get("source_y_m", 0.0)) != 0.0:
        raise ValueError("DEGADISx route assumes the source is centred on local y=0.")

    u0 = float(met["wind_speed_reference_m_s"])
    zref = float(met["wind_reference_height_m"])
    roughness = float(met["roughness_length_m"])
    stability = "ABCDEF".index(str(met["stability_class"]).upper()) + 1
    tamb = float(met["temperature_K"])
    pamb_atm = float(met["pressure_Pa"]) / 101325.0
    relhum = float(met.get("relative_humidity_pct", 50.0))
    source_temp = float(hg["source_temperature_K"])
    source_radius = float(hg["source_radius_m"])
    q = float(src["mass_rate_kg_s"])
    if q <= 0.0:
        raise ValueError("source.mass_rate_kg_s must be positive for DEGADISx.")
    if source_radius <= 0.0:
        raise ValueError("heavy_gas.source_radius_m must be positive.")
    if tamb <= 0.0 or source_temp <= 0.0 or pamb_atm <= 0.0:
        raise ValueError("Temperature and pressure must be positive absolute values.")

    mw_g_mol = float(gas["molecular_weight_g_mol"])  # numerically kg/kmol for DEGADIS
    rho_source = (float(met["pressure_Pa"]) * (mw_g_mol / 1000.0) /
                  (R_UNIVERSAL * source_temp))
    cp_mass = float(gas["cp_molar_j_mol_k"]) / (mw_g_mol / 1000.0)
    ulc, llc, flammability_note = _degadisx_flammability_limits(cfg)
    yclow = min(float(hg["mole_fraction_cutoff"]), llc)
    avtime = float(hg["averaging_time_s"])
    radius = source_radius
    # The last two rows are zero-rate terminators. DEGADISx documents TEND as
    # the third-from-last source time; hence row 2 is the true release end.
    if cfg["model_mode"] == "continuous":
        tend = 1.0e5
        dt = 1.0
        rows = [
            (0.0, q, radius, 1.0, source_temp, 0.0),
            (tend, q, radius, 1.0, source_temp, 0.0),
            (tend + dt, 0.0, radius, 1.0, source_temp, 0.0),
            (tend + 2*dt, 0.0, radius, 1.0, source_temp, 0.0),
        ]
        steady_flag = "T"
    else:
        tend = float(src["release_duration_s"])
        if tend <= 0.0:
            raise ValueError("source.release_duration_s must be positive in puff mode.")
        dt = max(1.0, min(5.0, tend / 20.0))
        rows = [
            (0.0, q, radius, 1.0, source_temp, 0.0),
            (tend, q, radius, 1.0, source_temp, 0.0),
            (tend + dt, 0.0, radius, 1.0, source_temp, 0.0),
            (tend + 2*dt, 0.0, radius, 1.0, source_temp, 0.0),
        ]
        steady_flag = "F"

    lines = [
        "ChatGPT Colab DEGADISx dense-gas case",
        "Resolved low-momentum ground area source; flat terrain",
        f"Mixture: {gas['name'][:60]}",
        "DEGADISx 0.1.4 / DEGADIS 2.1-compatible input",
        f"{u0:.8g} {zref:.8g} {roughness:.8g}",
        f"{stability:d}",
        f"0.0 {avtime:.8g}",
        "1 0.0",  # use stability-dependent Monin-Obukhov defaults
        f"{tamb:.8g} {pamb_atm:.8g} 0.0 {relhum:.8g}",
        f"0 {tamb:.8g}",  # adiabatic source, surface temperature = ambient
        "0 0.0",            # no explicit sensible heat-transfer coefficient
        "0 0.0",            # no explicit ground water-transfer coefficient
        "MIX",              # 3-character pseudo-component label
        f"{mw_g_mol:.8g} {source_temp:.8g} {rho_source:.8g}",
        f"{cp_mass:.8g} 0.0",  # CPP=0: CPK is constant mass Cp in J/(kg K)
        f"{ulc:.12g} {llc:.12g} {float(cfg['domain']['receptor_height_m']):.8g}",
        f"{yclow:.12g}",
        "0.0",               # no initial contaminant mass on the source
        f"{len(rows)}",
    ]
    for row in rows:
        lines.append(" ".join(f"{v:.12g}" for v in row))
    lines += [steady_flag, "COLAB-MODEL"]
    Path(deck_path).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return {"deck_path": str(deck_path), "source_rows": rows, "source_rate_kg_s": q,
            "source_radius_m": radius, "llc": llc, "ulc": ulc,
            "flammability_note": flammability_note, "steady_flag": steady_flag}


def _table_cc_to_mole_fraction(cc_kg_m3, table):
    """Vectorized equivalent of DEGADISx AdiabaticTable.from_concentration.

    The dense-gas profile gives contaminant mass concentration (kg/m3); the
    display field is converted to contaminant mole fraction using the same
    tabulated density/mixing relation as the package's receptor calculation.
    """
    cc = np.maximum(np.asarray(cc_kg_m3, dtype=float), 0.0)
    rho = np.interp(cc, np.asarray(table.cc), np.asarray(table.rho))
    wc = np.divide(cc, rho, out=np.zeros_like(cc), where=rho > 0.0)
    humid = float(table.humid)
    humsrc = float(table.humsrc)
    wa = (1.0 - (1.0 + humsrc) * wc) / (1.0 + humid)
    ww = 1.0 - wa - wc
    mw = 1.0 / (wc / float(table.gasmw) + wa / 28.9647 + ww / 18.01528)
    yc = mw / float(table.gasmw) * wc
    return np.clip(yc, 0.0, 1.0)


def _degadisx_snapshot_to_3d(snapshot, source_result, cfg):
    """Reconstruct a 3D display grid from one DEGADISx dense-gas snapshot."""
    hg = cfg["heavy_gas"]
    max_x = float(cfg["domain"]["x_max_m"])
    dist = np.asarray(snapshot.column("dist"), dtype=float)
    cc = np.asarray(snapshot.column("ccstr"), dtype=float)
    b = np.asarray(snapshot.column("b"), dtype=float)
    sy = np.asarray(snapshot.column("sy"), dtype=float)
    sz = np.asarray(snapshot.column("sz"), dtype=float)
    valid = np.isfinite(dist) & np.isfinite(cc) & (dist >= 0.0)
    if np.count_nonzero(valid) < 2:
        raise RuntimeError("DEGADISx snapshot has fewer than two valid downwind profile points.")
    dist, cc, b, sy, sz = [v[valid] for v in (dist, cc, b, sy, sz)]
    order = np.argsort(dist)
    dist, cc, b, sy, sz = [v[order] for v in (dist, cc, b, sy, sz)]
    dist, unique_idx = np.unique(dist, return_index=True)
    cc, b, sy, sz = [v[unique_idx] for v in (cc, b, sy, sz)]
    if len(dist) < 2:
        raise RuntimeError("DEGADISx snapshot has an insufficient unique downwind profile.")

    xmax = min(max_x, float(np.max(dist)))
    if xmax <= 0.0:
        raise RuntimeError("DEGADISx snapshot contains no positive downwind extent.")
    xg = np.linspace(0.0, xmax, int(hg["nx"]))
    yg = np.linspace(-float(hg["y_half_width_m"]), float(hg["y_half_width_m"]), int(hg["ny"]))
    zg = np.linspace(0.0, float(hg["z_max_m"]), int(hg["nz"]))
    ccx = np.interp(xg, dist, cc, left=0.0, right=0.0)
    bx = np.maximum(np.interp(xg, dist, b, left=0.0, right=0.0), 0.0)
    syx = np.maximum(np.interp(xg, dist, sy, left=1e-9, right=1e-9), 1e-9)
    szx = np.maximum(np.interp(xg, dist, sz, left=1e-9, right=1e-9), 1e-9)
    alpha1 = float(source_result.alpha) + 1.0

    abs_y = np.abs(yg)[None, :]
    outside = abs_y > bx[:, None]
    lateral = np.ones((len(xg), len(yg)), dtype=float)
    lateral[outside] = np.exp(-(((abs_y - bx[:, None]) / syx[:, None]) ** 2)[outside])
    vertical = np.exp(-((zg[None, :] / szx[:, None]) ** alpha1))
    cc_grid = ccx[:, None, None] * lateral[:, :, None] * vertical[:, None, :]
    yc_grid = _table_cc_to_mole_fraction(cc_grid, source_result.thermo.table)
    ppm_grid = yc_grid * 1.0e6
    return xg, yg, zg, ppm_grid, yc_grid, {
        "dist_m": dist, "cc_centerline_kg_m3": cc, "b_m": b,
        "sy_m": sy, "sz_m": sz, "alpha1": alpha1,
        "profile_extent_m": float(np.max(dist)),
    }


def run_degadisx_engine(cfg):
    """Run DEGADISx on the configured resolved ground-level direct source."""
    run_steady, run_transient, Receptor = _degadisx_imports()
    with tempfile.TemporaryDirectory(prefix="degadisx_colab_") as tmp:
        deck = Path(tmp) / "COLAB_DENSE_GAS.INP"
        deck_meta = _write_degadisx_inp(deck, cfg)
        if cfg["model_mode"] == "continuous":
            profile, source_result = run_steady(deck)
            snapshot = None
            out = None
            prof_dist = profile.column("dist")
            prof_cc = profile.column("cc")
            prof_b = profile.column("b")
            prof_sy = profile.column("sy")
            prof_sz = profile.column("sz")
            class _SteadySnapshot:
                def column(self, key):
                    return {"dist": prof_dist, "ccstr": prof_cc, "b": prof_b,
                            "sy": prof_sy, "sz": prof_sz}[key]
            snapshot = _SteadySnapshot()
            result_kind = "steady"
        else:
            out = run_transient(deck)
            if not out.snapshots:
                raise RuntimeError("DEGADISx transient run returned no snapshots. Check source duration/rate and cutoff.")
            selection = cfg["heavy_gas"]["snapshot_selection"]
            if selection == "peak_centerline":
                def _peak(snap):
                    arr = np.asarray(snap.column("yc"), dtype=float)
                    return float(np.nanmax(arr)) if arr.size else -np.inf
                snapshot = max(out.snapshots, key=_peak)
            else:
                requested = float(cfg["heavy_gas"]["snapshot_time_s"])
                snapshot = min(out.snapshots, key=lambda snap: abs(float(snap.time) - requested))
            profile = None
            source_result = out.source
            result_kind = "transient"

        xg, yg, zg, ppm, mole, profile_meta = _degadisx_snapshot_to_3d(snapshot, source_result, cfg)
        receptor_df = None
        if result_kind == "transient" and cfg["heavy_gas"].get("run_receptor_histories", True):
            receptors = []
            for r in RECEPTORS:
                yloc = abs(float(r["y_m"]))  # DEGADISx off-axis profile is symmetric about centreline.
                receptors.append(Receptor(x=float(r["x_m"]), offsets=[(yloc, float(r["z_m"]))]))
            histories = out.dose(receptors, maxnt=80)
            hist_rows = []
            for r, hist in zip(RECEPTORS, histories):
                times = hist.column("time") if len(hist.rows) else np.array([])
                if len(times):
                    # Off-axis history is in mole fraction; centreline column is also included.
                    vals = hist.offaxis[:, 0] if hist.offaxis.size else hist.column("yc")
                    for t, c in zip(times, vals):
                        hist_rows.append({"Receptor": r["name"], "Time (s)": float(t),
                                          "Concentration (ppmv)": float(c) * 1.0e6,
                                          "Peak concentration (ppmv)": float(np.max(vals)) * 1.0e6,
                                          "Peak time (s)": float(times[int(np.argmax(vals))])})
            receptor_df = pd.DataFrame(hist_rows)
        return {
            "kind": result_kind, "profile": profile, "output": out,
            "source_result": source_result, "snapshot": snapshot,
            "x": xg, "y": yg, "z": zg, "ppm": ppm, "mole": mole,
            "profile_meta": profile_meta, "deck_meta": deck_meta,
            "receptor_history": receptor_df,
        }


if CONFIG["model_engine"] == "degadisx_ground":
    print("Running DEGADISx dense-gas engine. The Gaussian results above are retained as a passive-model comparator.")
    DEGADISX_RESULT = run_degadisx_engine(CONFIG)
    DEGADISX_PROFILE = DEGADISX_RESULT["profile"]
    DEGADISX_SOURCE_RESULT = DEGADISX_RESULT["source_result"]
    DEGADISX_SNAPSHOT = DEGADISX_RESULT["snapshot"]
    DEGADISX_X, DEGADISX_Y, DEGADISX_Z = (DEGADISX_RESULT["x"], DEGADISX_RESULT["y"], DEGADISX_RESULT["z"])
    DEGADISX_FIELD_PPM = DEGADISX_RESULT["ppm"]
    DEGADISX_FIELD_MOLE = DEGADISX_RESULT["mole"]
    DEGADISX_HISTORY_DF = DEGADISX_RESULT["receptor_history"]
    print(f"DEGADISx mode: {DEGADISX_RESULT['kind']}; source radius={DEGADISX_RESULT['deck_meta']['source_radius_m']:.3g} m; source rate={DEGADISX_RESULT['deck_meta']['source_rate_kg_s']:.4g} kg/s")
    print(f"Snapshot time: {getattr(DEGADISX_SNAPSHOT, 'time', float('nan')):.3g} s; maximum field concentration={np.nanmax(DEGADISX_FIELD_PPM):.4g} ppmv")
    print("Flammability handling:", DEGADISX_RESULT["deck_meta"]["flammability_note"])
    print("Profile output extent:", f"{DEGADISX_RESULT['profile_meta']['profile_extent_m']:.1f} m")

    # 3D isosurface (post-processed visualization of DEGADISx profile shape).
    iso = float(CONFIG["heavy_gas"]["plot_isosurface_level_ppm"])
    if np.nanmax(DEGADISX_FIELD_PPM) > iso:
        XX, YY, ZZ = np.meshgrid(DEGADISX_X, DEGADISX_Y, DEGADISX_Z, indexing="ij")
        fig_hg_3d = go.Figure(go.Isosurface(
            x=XX.ravel(), y=YY.ravel(), z=ZZ.ravel(), value=DEGADISX_FIELD_PPM.ravel(),
            isomin=iso, isomax=float(np.nanmax(DEGADISX_FIELD_PPM)), surface_count=3,
            caps=dict(x_show=False, y_show=False, z_show=False),
            colorbar=dict(title="DEGADISx ppmv"), opacity=0.55,
        ))
        fig_hg_3d.update_layout(title="DEGADISx dense-gas field — 3D profile reconstruction",
            scene=dict(xaxis_title="Downwind x (m)", yaxis_title="Crosswind y (m)", zaxis_title="Height z (m)"), height=700)
        fig_hg_3d.show()
    else:
        print(f"3D isosurface not plotted: maximum concentration is below the {iso:g} ppm display threshold.")

    iz_rec = int(np.argmin(np.abs(DEGADISX_Z - float(CONFIG["domain"]["receptor_height_m"]))))
    fig_hg_ground = go.Figure(go.Contour(
        x=DEGADISX_X, y=DEGADISX_Y, z=DEGADISX_FIELD_PPM[:, :, iz_rec].T,
        colorscale="Viridis", colorbar=dict(title="Total mixture ppmv"),
        contours=dict(coloring="fill"),
        hovertemplate="x=%{x:.1f} m<br>y=%{y:.1f} m<br>C=%{z:.4g} ppmv<extra></extra>"))
    fig_hg_ground.update_layout(title="DEGADISx dense-gas ground/receptor-height concentration — profile reconstruction",
        xaxis_title="Downwind x (m)", yaxis_title="Crosswind y (m)", height=600)
    fig_hg_ground.show()

    iy0 = int(np.argmin(np.abs(DEGADISX_Y)))
    fig_hg_xz = go.Figure(go.Contour(
        x=DEGADISX_X, y=DEGADISX_Z, z=DEGADISX_FIELD_PPM[:, iy0, :].T,
        colorscale="Viridis", colorbar=dict(title="Total mixture ppmv"),
        contours=dict(coloring="fill"),
        hovertemplate="x=%{x:.1f} m<br>z=%{y:.2f} m<br>C=%{z:.4g} ppmv<extra></extra>"))
    fig_hg_xz.update_layout(title="DEGADISx vertical centreline concentration — profile reconstruction",
        xaxis_title="Downwind x (m)", yaxis_title="Height z (m)", height=560)
    fig_hg_xz.show()

    # Component-resolved concentration summary using fixed release composition.
    dense_component_rows = []
    max_total = float(np.nanmax(DEGADISX_FIELD_PPM))
    for name, yi in CONFIG["gas"]["composition_mole_fractions"].items():
        rec = CHEMICAL_DATABASE[name]
        endpoint, window_s, label, note = selected_toxicity_endpoint(rec, CONFIG["hazard_assessment"]["toxicity_endpoint"])
        dense_component_rows.append({
            "Component": name,
            "Released-gas mole fraction (%)": 100.0 * float(yi),
            "Maximum component concentration (ppmv)": max_total * float(yi),
            "LFL (vol% in air)": rec.get("lfl_vol_pct"),
            "UFL (vol% in air)": rec.get("ufl_vol_pct"),
            "Selected toxic endpoint": label,
            "Endpoint (ppm)": endpoint,
            "Endpoint window (s)": window_s,
            "Data note": note,
        })
    dense_component_df = pd.DataFrame(dense_component_rows)
    print("DEGADISx component concentration / hazard reference summary (fixed composition assumption):")
    display(dense_component_df)

    if DEGADISX_HISTORY_DF is not None and not DEGADISX_HISTORY_DF.empty:
        print("DEGADISx receptor time histories (component mix fraction held constant):")
        display(DEGADISX_HISTORY_DF)
    elif CONFIG["model_mode"] == "continuous":
        print("Steady DEGADISx run selected; time-dependent receptor histories are not available for a steady plume.")
else:
    print("DEGADISx engine not selected. The passive Gaussian results above remain the active model outputs.")

# Synthetic unit test of the DEGADISx profile-to-field reconstruction helper.
# This verifies the 3D post-processing shape and array dimensions without running the external solver.
class _SyntheticProfile:
    def __init__(self):
        self.alpha = 0.15
        class _Table:
            cc = np.array([0.0, 0.01, 0.1, 1.0])
            rho = np.array([1.18, 1.17, 1.12, 0.9])
            humid = 0.0
            humsrc = 0.0
            gasmw = 44.0
        self.thermo = type("_Thermo", (), {"table": _Table()})()

class _SyntheticSnapshot:
    time = 30.0
    def column(self, key):
        values = {
            "dist": np.array([5.0, 10.0, 25.0, 50.0]),
            "ccstr": np.array([0.02, 0.015, 0.006, 0.001]),
            "b": np.array([2.0, 1.8, 0.5, 0.0]),
            "sy": np.array([1.0, 2.0, 4.0, 8.0]),
            "sz": np.array([0.5, 0.8, 1.5, 3.0]),
        }
        return values[key]

_test_cfg = {"domain": {"x_max_m": 100.0}, "heavy_gas": {"nx": 25, "ny": 21, "nz": 17, "y_half_width_m": 20.0, "z_max_m": 20.0}}
_test = _degadisx_snapshot_to_3d(_SyntheticSnapshot(), _SyntheticProfile(), _test_cfg)
assert _test[3].shape == (25, 21, 17)
assert np.all(np.isfinite(_test[3])) and np.nanmax(_test[3]) > 0
print("DEGADISx 3D post-processing synthetic check passed.")


## 15. Built-in numerical checks

These tests check implementation consistency, not experimental validation. They cover wind-profile normalization, grid stretching, dispersion coefficients, pressure-vessel blowdown monotonicity, and finite-duration source integration and source-plane continuity.

In [ ]:
# 1) Reference wind speed is recovered at the measurement height for every stability class.
for stability in "ABCDEF":
    met_test = dict(CONFIG["meteorology"])
    met_test["stability_class"] = stability
    u_ref = wind_velocity(met_test["wind_reference_height_m"], met_test)
    assert np.isclose(u_ref, met_test["wind_speed_reference_m_s"], rtol=1e-8), (stability, u_ref)

# 2) Dispersion parameters are positive and finite.
for stability in "ABCDEF":
    for terrain in ("Rural", "Urban"):
        ss = dispersion_sigmas(np.array([1.0, 10.0, 100.0, 1000.0]), stability, terrain)
        assert all(np.all(np.isfinite(v)) and np.all(v > 0) for v in ss)

# 3) Stretched grid is monotonic, nonuniform and includes the receptor height.
assert np.all(np.diff(x) > 0) and np.all(np.diff(y) > 0) and np.all(np.diff(z) > 0)
assert np.any(np.isclose(z, CONFIG["domain"]["receptor_height_m"]))
assert np.std(np.diff(x)) > 0 and np.std(np.diff(y)) > 0 and np.std(np.diff(z)) > 0

# 4) Exact source plane, source crosswind coordinate and release height are in the grid.
assert np.isclose(x[0], 0.0)
assert np.any(np.isclose(y, CONFIG["source"]["source_y_m"]))
assert np.any(np.isclose(z, CONFIG["source"]["release_height_m"]))

# 5) Transient finite-duration source remains nonzero at the source while it is active,
# and decays to numerical zero at the source after shutoff for this finite-duration case.
if CONFIG["model_mode"] == "puff":
    Hs = float(CONFIG["source"]["release_height_m"])
    ys = float(CONFIG["source"]["source_y_m"])
    t_test_active = min(0.5 * RELEASE_SUMMARY["release_duration_s"],
                        max(1.0, CONFIG["source"].get("release_duration_s", 60.0) * 0.5))
    c_active = transient_gaussian_receptor_concentration(
        (0.0, ys, Hs), t_test_active, CONFIG,
        SEGMENT_STARTS, SEGMENT_ENDS, SEGMENT_RATES)
    assert c_active > 0.0, "Active finite-duration source should have nonzero concentration at x=0 on the source plane."
    if t_test_active < RELEASE_SUMMARY["release_duration_s"]:
        c_after = transient_gaussian_receptor_concentration(
            (0.0, ys, Hs), RELEASE_SUMMARY["release_duration_s"] + 60.0,
            CONFIG, SEGMENT_STARTS, SEGMENT_ENDS, SEGMENT_RATES)
        assert c_after <= max(c_active * 1e-6, 1e-18), "After shutoff, source-plane concentration should decay strongly."

# 5) Optional tank blowdown check using a copy of the input with vessel source enabled.
_test_cfg = {k: (v.copy() if isinstance(v, dict) else v) for k,v in CONFIG.items()}
_test_cfg["source"] = {k: (v.copy() if isinstance(v, dict) else v) for k,v in CONFIG["source"].items()}
_test_cfg["source"]["source_term_mode"] = "vessel_gas_blowdown"
try:
    test_prof, test_meta = vessel_blowdown_profile(_test_cfg)
    assert np.all(np.diff(test_prof.tank_pressure_bar_abs) <= 1e-8)
    assert np.all(np.diff(test_prof.tank_mass_kg) <= 1e-8)
    assert np.all(test_prof.mass_rate_kg_s >= 0)
    assert test_meta["released_mass_kg"] >= 0
except ValueError as exc:
    # A scenario-specific input could make the optional vessel test inapplicable.
    print("Optional vessel test skipped because vessel inputs are not applicable:", exc)

# 6) Computed snapshot is finite and nonnegative.
assert np.all(np.isfinite(C_kg_m3)) and np.min(C_kg_m3) >= 0
print("All numerical checks passed.")
print(f"Source-plane concentration during active release: {c_active:.5g} kg/m³ (after shutoff: {c_after:.5g} kg/m³ when tested).")

# 7) Chemical database completeness and nonnegative component fractions.
assert len(CHEMICAL_DATABASE) >= 28
_mix_test = calculate_gas_mixture({"fraction_basis": "mole", "components": [
    {"chemical": "Hydrogen cyanide (HCN)", "fraction": 0.25},
    {"chemical": "Nitrogen (N2)", "fraction": 0.75},
]})
assert np.isclose(sum(_mix_test["composition_mole_fractions"].values()), 1.0)
assert _mix_test["molecular_weight_g_mol"] > 0


# 8) Chemical database coverage and conservative missing-data handling.
assert "1,3-Butadiene (C4H6)" in CHEMICAL_DATABASE
assert "Methyl mercaptan (CH3SH)" in CHEMICAL_DATABASE
assert "Phosphine (PH3)" in CHEMICAL_DATABASE
assert "Phosgene (COCl2)" in CHEMICAL_DATABASE
assert "Nitric oxide (NO)" in CHEMICAL_DATABASE
assert CHEMICAL_DATABASE["Phosphine (PH3)"].get("ufl_vol_pct") is None
assert CHEMICAL_DATABASE["Phosphine (PH3)"].get("diffusion_volume_cm3_mol") is None

_ph3_mix = calculate_gas_mixture({"fraction_basis": "mole", "components": [
    {"chemical": "Phosphine (PH3)", "fraction": 0.01},
    {"chemical": "Nitrogen (N2)", "fraction": 0.99},
]})
_ph3_haz = calculate_mixture_hazard_properties(
    {"fraction_basis": "mole"}, _ph3_mix, CONFIG["meteorology"]
)
assert _ph3_haz["mixed_fuel_lfl_vol_pct"] is None
assert "Phosphine" in _ph3_haz["flammability_rule"]
assert np.isnan(fuller_diffusivity_in_air_m2_s(
    CHEMICAL_DATABASE["Phosphine (PH3)"], 298.15, 101325.0
))
print("Chemical database checks passed (28 species).")


## 16. Model assumptions, limitations, and validation guidance

### Model formulation and available capabilities

The model combines chemical-mixture properties, source-term calculations, atmospheric meteorology, passive Gaussian dispersion, an optional DEGADISx dense-gas pathway, component-specific hazard screening, and visualisation/export functions.

- **Atmospheric wind:** a stability-dependent power-law profile is available as the default, with an optional Monin–Obukhov-type surface-layer profile. Both are normalised so that the entered wind speed is recovered at the reference height. The Gaussian transport velocity is vertically weighted using the reflected vertical concentration profile. The default power-law exponents are typical values documented for a roughness length of 0.15 m; site calibration is preferable.
- **Computational grid:** nonuniform `x`, `y`, and `z` coordinates provide finer resolution near the source, plume centreline, and ground. Exact source and receptor coordinates can be included. Grid-spacing diagnostics and direct-versus-interpolated receptor comparisons help identify possible under-resolution.
- **Transient dispersion:** the finite-duration Gaussian cloud formulation integrates the longitudinal Gaussian kernel over each active source interval using error functions. Constant-rate releases are represented as finite source intervals; time-varying vessel blowdown is represented by piecewise-constant source-rate segments. The source plane is included in the grid. During an active release, concentration may be non-zero at the source plane; after shutoff, the cloud is expected to move downwind.
- **Gas release:** direct user-specified mass rate, an initial constant-pressure ideal-gas orifice estimate, and rigid-vessel adiabatic ideal-gas blowdown are supported. The blowdown calculation updates vessel pressure, temperature, inventory, and discharge rate with time.
- **Chemical mixtures:** the chemical selector supports 28 curated species and mole- or mass-fraction input. Mixture molecular weight, ideal-gas Cp, Cv, gamma, and specific gas constant are calculated. The built-in hazard and transport records include component LFL/UFL where sourced, estimated gas-in-air molecular diffusivity, and available AEGL-2, ERPG-2, and IDLH values.
- **Flammability and toxicity screening:** the Le Chatelier rule provides an indicative mixed-fuel LFL/UFL when all selected combustible components have the required numerical limits. Component concentrations are compared individually with available toxicological endpoints. Snapshot hazard contours are instantaneous comparisons; receptor exposure-window screening is calculated only when the concentration history spans the full selected endpoint averaging period.
- **Dense-gas dispersion:** the optional `degadisx_ground` pathway calls DEGADISx 0.1.4, a Python reimplementation of U.S. EPA DEGADIS 2.1. The integrated route is limited to a low-momentum ground-level area source over flat, level terrain. The source rate and radius must describe the resolved ground-level vapour source after any upstream jet, flashing, rainout, or pool calculation. DEGADISx reports a downwind profile; the notebook reconstructs a Cartesian visualisation field from the reported centreline concentration, lateral core/shoulder dimensions, and vertical profile. This is a post-processing representation, not a native 3D DEGADIS or CFD solution.

### Important assumptions and limitations

1. The Gaussian dispersion engine represents passive gas dispersion. The Richardson number is an applicability screen only; it is not a dense-gas solver.
2. The default ALOHA power-law wind exponents are typical values documented for a roughness length of 0.15 m. If the actual site roughness differs substantially, use site-calibrated profile data or set `power_law_n_override`; the optional Monin–Obukhov profile is an alternative simplified surface-layer representation.
3. Wind direction and stability are constant during a run. The wind profile is horizontally homogeneous; terrain steering, buildings, obstacles, and urban-canopy flow are not resolved.
4. Briggs/ALOHA Table 13 parameters are empirical and have measurement-averaging assumptions. Predictions should be checked against validation data and the relevant averaging time.
5. The transient Gaussian pathway uses the finite-duration error-function cloud formulation described in ALOHA Technical Documentation Section 4.3, adapted to use local effective wind from the selected vertical profile and piecewise-constant source-rate segments for time-varying blowdown. It is not a complete reproduction of ALOHA's internal algorithms and does not resolve source-jet near-field physics. The 1 m minimum distance used to regularise Briggs dispersion sigmas at x = 0 is a numerical point-source regularisation.
6. Vessel blowdown assumes a rigid, adiabatic, well-mixed vessel containing a single ideal gas or fixed-composition ideal-gas mixture with constant mixture heat-capacity ratio derived from ideal-gas Cp at 298.15 K. It excludes wall heat transfer, real-gas effects, liquefied gas, flashing, two-phase flow, aerosols, composition fractionation, and pipe losses.
7. The direct source-rate and initial orifice options are user-defined approximations. A constant-pressure orifice rate should not be mistaken for a pressure-vessel blowdown profile.
8. The Gaussian model becomes unreliable where gravity, jet momentum, buoyancy, or very-near-field effects are important. Do not use Gaussian hazard distances for dense liquefied-gas releases without a validated dense-gas model and a suitable source term.
9. The DEGADISx pathway is restricted to a resolved, low-momentum, ground-level area source on flat terrain. The notebook does not automatically calculate the high-pressure jet, jet trajectory, flashing, aerosol rainout, pool spread, or pool evaporation needed to derive such a source.
10. The DEGADISx package is a beta, research-oriented reimplementation. A successful notebook execution does not establish independent validation of its predictions. The reconstructed 3D dense-gas field is for visualisation and receptor interpolation only.
11. Concentration thresholds used in plots are visualisation levels only. They are not automatically AEGL, ERPG, IDLH, LFL, or UFL values.
12. The embedded database is a curated dataset. Molecular weights and gas-phase heat capacities are linked to NIST Chemistry WebBook/CCCBDB; flammability and toxicological endpoints are populated only where a cited NOAA CAMEO, EPA, or NIOSH source provides a value. Fuller diffusion-volume entries identified as atomic/group-increment estimates are not measured binary diffusion coefficients. Missing fields mean unavailable in the curated source set, not zero or safe.
13. Heat capacities are treated as constant values near 298.15 K, mixture composition remains fixed during release and dispersion, and gas mixtures are assumed ideal. Non-ideal interactions, condensation, selective evaporation, chemical reaction, and preferential sorption are not represented.
14. Toxic endpoint snapshot contours are instantaneous concentration overlays, not duration-averaged exposure contours. Receptor moving-window screening is performed only when the time history spans the complete endpoint averaging period (3600 s for AEGL-2/ERPG-2 and 1800 s for the IDLH screening window). The result is a screening comparison, not a complete regulatory exposure model.
15. For propane and n-butane, EPA AEGL-2 values can lie above 50% of the EPA-stated LEL; the database retains the warning that explosion hazards must also be considered. DME hazard values vary among compilations; verify the selected record and its source before use.

### Recommended validation before engineering use

- Verify the passive Gaussian model against tracer field experiments such as Prairie Grass using centreline and crosswind concentration profiles.
- Compare wind-profile calculations against site meteorological data or a recognised boundary-layer formulation.
- Check release-rate histories against an independent compressible-flow/blowdown calculation for the same gas, vessel, and orifice conditions.
- Perform grid-refinement studies by increasing `nx`, `ny`, and `nz` and comparing peak and receptor concentrations, contour distances, and cloud dimensions.
- Verify the actual DEGADISx backend using documented test cases or appropriate dense-gas field experiments. Check the external solver separately from the notebook's 3D post-processing reconstruction.
- Audit each chemical property against its current primary source. Record source edition/date, units, exposure period, interim/final status, and the basis for any IDLH value.
- Document the source model, meteorological basis, grid-convergence outcome, benchmark comparison, and unresolved scenario physics for every study case.

### Chemical and model reference basis

The 28-species library uses NIST Chemistry WebBook (SRD 69) and NIST CCCBDB for molecular-weight and gas-phase heat-capacity records; NOAA CAMEO Chemicals, NIOSH Pocket Guide, and U.S. EPA AEGL resources for chemical-specific flammability and toxicological entries where values are available; and Fuller-Schettler-Giddings for estimated binary gas-phase diffusivities. Individual chemical links are displayed in the database tables.

Key model references include:

- Jones, R., Lehr, W., Simecek-Beatty, D., & Reynolds, R. M. (2013). *ALOHA (Areal Locations of Hazardous Atmospheres) 5.4.4: Technical Documentation*. NOAA Technical Memorandum NOS OR&R 43. Sections 4.2–4.4 and Table 13.
- Palazzi, E., De Faveri, M., Fumarola, G., & Ferraiolo, G. (1982). Diffusion from a steady source of short duration. *Atmospheric Environment*, 16(12), 2785–2790.
- Briggs, G. A. (1973). *Diffusion estimation for small emissions: Preliminary draft*. Atmospheric Turbulence and Diffusion Laboratory.
- Fuller, E. N., Schettler, P. D., & Giddings, J. C. (1966). A new method for prediction of binary gas-phase diffusion coefficients. *Industrial & Engineering Chemistry*, 58(5), 18–27.
- Spicer, T. O., & Havens, J. A. (1989). *User's Guide for the DEGADIS 2.1 Dense Gas Dispersion Model* (EPA-450/4-89-019). U.S. Environmental Protection Agency.
- DEGADISx 0.1.4, Python reimplementation of U.S. EPA DEGADIS 2.1; package and source links are provided in the references below.

### References

- Beals, G. A. (1971). *Guide to local diffusion of air pollutants*. Air Weather Service (MAC), U.S. Air Force.
- Briggs, G. A. (1973). *Diffusion estimation for small emissions: Preliminary draft*. Atmospheric Turbulence and Diffusion Laboratory, Oak Ridge, Tennessee.
- Britter, R. E. (1989). Atmospheric dispersion of dense gases. *Annual Review of Fluid Mechanics*, 21, 317–344.
- Brutsaert, W. (1982). *Evaporation into the atmosphere: Theory, history, and applications*. D. Reidel.
- Businger, J. A. (1973). Turbulence transfer in the atmospheric surface layer. In D. A. Haugen (Ed.), *Workshop on Micrometeorology*. American Meteorological Society.
- Colenbrander, G. W. (1980). A mathematical model for the transient behavior of dense vapor clouds. Proceedings of the 3rd International Symposium on Loss Prevention and Safety Promotion in the Process Industries, Basel, Switzerland.
- Colenbrander, G. W., & Puttock, J. S. (1983). Dense gas dispersion behavior: Experimental observations and model developments. International Symposium on Loss Prevention and Safety Promotion in the Process Industries, Harrogate, England.
- Fuller, E. N., Schettler, P. D., & Giddings, J. C. (1966). A new method for prediction of binary gas-phase diffusion coefficients. *Industrial & Engineering Chemistry*, 58(5), 18–27. https://doi.org/10.1021/ie50677a007
- Hanna, S. R., Briggs, G. A., & Hosker, R. P. (1982). *Handbook on atmospheric diffusion*. U.S. Department of Energy, Technical Information Center.
- Jones, R., Lehr, W., Simecek-Beatty, D., & Reynolds, R. M. (2013). *ALOHA (Areal Locations of Hazardous Atmospheres) 5.4.4: Technical Documentation*. NOAA Technical Memorandum NOS OR&R 43. U.S. Department of Commerce / NOAA. https://repository.library.noaa.gov/view/noaa/2669
- Lee, U. (2026). *degadisx* (Version 0.1.4) [Python software]. PyPI. https://pypi.org/project/degadisx/0.1.4/
- Lee, U. (2026). *degadisx* [Source code]. GitHub. https://github.com/lyullee/degadisx
- National Institute of Standards and Technology. (2025). *NIST Chemistry WebBook*, SRD 69. https://doi.org/10.18434/T4D303
- National Institute of Standards and Technology. (n.d.). *Computational Chemistry Comparison and Benchmark Database (CCCBDB)*. https://cccbdb.nist.gov/
- NOAA Office of Response and Restoration. (n.d.). *CAMEO Chemicals*. https://cameochemicals.noaa.gov/
- Palazzi, E., De Faveri, M., Fumarola, G., & Ferraiolo, G. (1982). Diffusion from a steady source of short duration. *Atmospheric Environment*, 16(12), 2785–2790.
- Pasquill, F. (1961). The estimation of the dispersion of windborne material. *Meteorological Magazine*, 90, 33–49.
- Shapiro, A. H. (1953). *The dynamics and thermodynamics of compressible fluid flow* (Vol. I). Wiley.
- Spicer, T. O., & Havens, J. A. (1989). *User's guide for the DEGADIS 2.1 dense gas dispersion model* (EPA-450/4-89-019). U.S. Environmental Protection Agency. https://nepis.epa.gov/Exe/ZyPURL.cgi?Dockey=2000J5GU.TXT
- U.S. Environmental Protection Agency. (n.d.). *Air quality dispersion modeling: Alternative models (DEGADIS and SLAB)*. https://www.epa.gov/scram/air-quality-dispersion-modeling-alternative-models
- U.S. Environmental Protection Agency. (n.d.). *About Acute Exposure Guideline Levels (AEGLs)*. https://www.epa.gov/aegl/about-acute-exposure-guideline-levels-aegls
- U.S. Environmental Protection Agency. (n.d.). *Access Acute Exposure Guideline Levels (AEGLs) values*. https://www.epa.gov/aegl/access-acute-exposure-guideline-levels-aegls-values
- VaibhavVasdev. (n.d.). *Gaussian-Plume_Model* [GitHub repository]. https://github.com/VaibhavVasdev/Gaussian-Plume_Model
- National Institute for Occupational Safety and Health. (n.d.). *NIOSH Pocket Guide to Chemical Hazards*. Centers for Disease Control and Prevention. https://www.cdc.gov/niosh/npg/npg.html
- COMSOL AB. (2024). *Transport properties: Gas diffusivity correlations* (COMSOL Multiphysics 6.4 documentation). https://doc.comsol.com/6.4/doc/com.comsol.help.lgp/lgp_ug_thermo.5.27.html

**Do not use this notebook alone to establish emergency exclusion zones, emergency response distances, or safety-system design limits.** Independent model validation and scenario-specific engineering review are required.